In [2]:
import sys
sys.path.append('/home/jovyan/work/scripts')

try:
    from database_checker import DatabaseChecker
    print("✅ Модуль найден!")
    print("   Путь:", __import__('database_checker').__file__)
except ImportError as e:
    print(f"❌ Ошибка: {e}")
    print("\nСодержимое scripts/:")
    !ls -la /home/jovyan/work/scripts/

✅ Модуль найден!
   Путь: /home/jovyan/work/scripts/database_checker.py


In [ ]:
#https://repository.econdata.tech/organization/rosstat   ---сайт россстат

In [3]:
#создание файла
%%writefile /home/jovyan/work/dags/sales_forecast_dag.py

UsageError: Line magic function `%%writefile` not found.


In [4]:
# В Jupyter выполнить эту ячейку
weather_dag_content = '''"""
DAG для ежедневного сбора погоды через Open-Meteo
"""

from airflow import DAG
from airflow.operators.python import PythonOperator
from datetime import datetime, timedelta
import sys
sys.path.append('/opt/airflow/scripts')
from weather_collector import collect_all_cities_weather

default_args = {
    'owner': 'data_engineer',
    'depends_on_past': False,
    'start_date': datetime(2024, 1, 1),
    'email_on_failure': True,
    'retries': 2,
    'retry_delay': timedelta(minutes=5),
}

def collect_yesterday_weather(**context):
    """Собирает погоду за вчерашний день"""
    yesterday = (datetime.now() - timedelta(days=1)).strftime('%Y-%m-%d')
    collect_all_cities_weather(yesterday, yesterday)

def collect_historical_2025(**context):
    """Собирает историю за 2025 год (запустить один раз)"""
    collect_all_cities_weather('2025-01-01', '2025-12-31')

with DAG(
    'weather_collection_daily',
    default_args=default_args,
    description='Ежедневный сбор погоды по городам',
    schedule_interval='0 8 * * *',
    catchup=False,
    tags=['weather', 'external_data'],
) as dag:

    task_daily = PythonOperator(
        task_id='collect_yesterday_weather',
        python_callable=collect_yesterday_weather
    )

    task_historical = PythonOperator(
        task_id='collect_historical_2025',
        python_callable=collect_historical_2025
    )
'''

# Сохраняем файл
with open('/home/jovyan/work/dags/weather_dag.py', 'w') as f:
    f.write(weather_dag_content)

print(" DAG файл создан")

 DAG файл создан


In [39]:
#сбор погоды
import requests
import urllib.parse
import time
import json
from clickhouse_driver import Client
import pandas as pd

# --- Конфигурация ---
CLICKHOUSE_HOST = 'my_clickhouse'
CLICKHOUSE_PORT = 9000
CLICKHOUSE_DB = 'external_data'
# -------------------

def get_cities_from_clickhouse():
    """
    Получает список уникальных городов из таблицы sales_raw в ClickHouse.
    """
    client = Client(
        host=CLICKHOUSE_HOST,
        port=CLICKHOUSE_PORT,
        user='default',
        password='',
        database=CLICKHOUSE_DB
    )
    
    try:
        query = """
        SELECT DISTINCT `город` as city
        FROM external_data.sales_raw
        WHERE `город` IS NOT NULL AND `город` != ''
        ORDER BY `город`
        """
        
        df_cities = client.query_dataframe(query)
        cities_list = df_cities['city'].tolist() if not df_cities.empty else []
        
        print(f"\n{'='*60}")
        print(f" ПОЛУЧЕНО ГОРОДОВ ИЗ CLICKHOUSE: {len(cities_list)}")
        print('='*60)
        
        if cities_list:
            # Покажем статистику
            stats_query = """
            SELECT 
                `город` as city,
                COUNT(*) as records_count,
                COUNT(DISTINCT `Номенклатура`) as products_count
            FROM external_data.sales_raw
            WHERE `город` IS NOT NULL AND `город` != ''
            GROUP BY `город`
            ORDER BY records_count DESC
            LIMIT 10
            """
            
            stats_df = client.query_dataframe(stats_query)
            print("\n Топ-10 городов по количеству записей:")
            for idx, row in stats_df.iterrows():
                print(f"   {row['city']}: {row['records_count']} записей, {row['products_count']} товаров")
            
            print(f"\n Всего городов: {len(cities_list)}")
            print(" Первые 10 городов по алфавиту:")
            for city in sorted(cities_list)[:10]:
                print(f"   - {city}")
        
        return cities_list
    
    except Exception as e:
        print(f" Ошибка при получении городов: {e}")
        return []

def get_coordinates_for_city(city_name):
    """
    Получает координаты города через OpenStreetMap Nominatim API
    """
    # Кодируем название города для URL
    url = f"https://nominatim.openstreetmap.org/search?q={urllib.parse.quote(city_name)}&format=json&limit=1"
    
    try:
        # Nominatim требует User-Agent
        headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
        }
        response = requests.get(url, headers=headers, timeout=10)
        data = response.json()
        
        if data:
            lat = float(data[0]['lat'])
            lon = float(data[0]['lon'])
            return {'lat': lat, 'lon': lon}
        else:
            return None
            
    except Exception as e:
        return None

def build_city_coordinates_dictionary(cities_list, save_to_file=True):
    """
    Строит словарь координат для списка городов и сохраняет в JSON
    """
    print(f"\n{'='*60}")
    print(f" ПОЛУЧЕНИЕ КООРДИНАТ ДЛЯ {len(cities_list)} ГОРОДОВ")
    print('='*60)
    
    city_coords = {}
    failed_cities = []
    
    for i, city in enumerate(cities_list, 1):
        print(f"\r {i}/{len(cities_list)}: {city[:30]:<30} ...", end="")
        
        coords = get_coordinates_for_city(city)
        
        if coords:
            city_coords[city] = coords
            print(f" ✓ {coords['lat']:.4f}, {coords['lon']:.4f}")
        else:
            failed_cities.append(city)
            print(f" ✗ не найдено")
        
        # Пауза между запросами
        time.sleep(1)
    
    print(f"\n{'='*60}")
    print(f" ПОЛУЧЕНИЕ КООРДИНАТ ЗАВЕРШЕНО:")
    print(f"   Успешно: {len(city_coords)} городов")
    print(f"   Не удалось: {len(failed_cities)} городов")
    
    if failed_cities:
        print("\n Города без координат:")
        for city in failed_cities[:20]:
            print(f"   - {city}")
        if len(failed_cities) > 20:
            print(f"   ... и еще {len(failed_cities) - 20}")
    
    # Сохраняем в JSON файл
    if save_to_file and city_coords:
        filename = 'city_coordinates.json'
        with open(filename, 'w', encoding='utf-8') as f:
            json.dump(city_coords, f, ensure_ascii=False, indent=2)
        print(f"\n Координаты сохранены в файл: {filename}")
        
        # Также сохраняем список неудачных городов
        if failed_cities:
            failed_filename = 'failed_cities.json'
            with open(failed_filename, 'w', encoding='utf-8') as f:
                json.dump(failed_cities, f, ensure_ascii=False, indent=2)
            print(f" Список городов без координат: {failed_filename}")
    
    return city_coords, failed_cities

def load_coordinates_from_file(filename='city_coordinates.json'):
    """
    Загружает координаты из JSON файла
    """
    try:
        with open(filename, 'r', encoding='utf-8') as f:
            city_coords = json.load(f)
        print(f"\n Загружены координаты для {len(city_coords)} городов из {filename}")
        return city_coords
    except FileNotFoundError:
        print(f"\n Файл {filename} не найден")
        return {}
    except Exception as e:
        print(f"\n Ошибка загрузки {filename}: {e}")
        return {}

# --- ВЫПОЛНЕНИЕ ---
print("="*60)
print(" ЭТАП 1: ПОЛУЧЕНИЕ КООРДИНАТ ГОРОДОВ")
print("="*60)

# Получаем города из ClickHouse
cities = get_cities_from_clickhouse()

if cities:
    # Получаем координаты
    city_coordinates, failed = build_city_coordinates_dictionary(cities)
    
    print(f"\n ИТОГО:")
    print(f"   Городов с координатами: {len(city_coordinates)}")
    print(f"   Городов без координат: {len(failed)}")
    
    # Покажем пример первых 5
    print("\n Примеры координат:")
    for city, coords in list(city_coordinates.items())[:5]:
        print(f"   '{city}': {{'lat': {coords['lat']}, 'lon': {coords['lon']}}},")

 ЭТАП 1: ПОЛУЧЕНИЕ КООРДИНАТ ГОРОДОВ

 ПОЛУЧЕНО ГОРОДОВ ИЗ CLICKHOUSE: 126

 Топ-10 городов по количеству записей:
   Воронеж: 160285 записей, 245 товаров
   Москва: 33588 записей, 186 товаров
   Ростов-на-Дону: 17742 записей, 204 товаров
   Волгоград: 17512 записей, 193 товаров
   Самара: 16821 записей, 209 товаров
   Брянск: 13762 записей, 156 товаров
   Тула: 12328 записей, 135 товаров
   Санкт-Петербург: 11400 записей, 88 товаров
   Краснодар: 11316 записей, 153 товаров
   Орел: 10719 записей, 129 товаров

 Всего городов: 126
 Первые 10 городов по алфавиту:
   - Адыге-Хабль аул.
   - Актау
   - Алматы
   - Армавир
   - Астрахань
   - Атырау
   - Ашхабад
   - Баку
   - Батайск
   - Бейлаган

 ПОЛУЧЕНИЕ КООРДИНАТ ДЛЯ 126 ГОРОДОВ
 1/126: Адыге-Хабль аул.               ... ✓ 44.3033, 41.7426
 2/126: Актау                          ... ✓ 43.6353, 51.1682
 3/126: Алматы                         ... ✓ 43.2364, 76.9457
 4/126: Армавир                        ... ✓ 44.9994, 41.1294
 5/126: Аст

In [1]:
#Сбор исторических данных о погоде
import requests
from datetime import datetime, timedelta
import time
from clickhouse_driver import Client
import json
from tqdm import tqdm

# --- Конфигурация ---
CLICKHOUSE_HOST = 'my_clickhouse'
CLICKHOUSE_PORT = 9000
CLICKHOUSE_DB = 'external_data'
CLICKHOUSE_TABLE = 'weather_hourly'  # оставляем существующую таблицу
# -------------------

def create_weather_table():
    """
    Создает таблицу для погодных данных, если она не существует
    Используем структуру, совместимую с существующей таблицей
    """
    client = Client(
        host=CLICKHOUSE_HOST,
        port=CLICKHOUSE_PORT,
        user='default',
        password='',
        database=CLICKHOUSE_DB
    )
    
    # Проверяем существование таблицы
    check_query = f"EXISTS TABLE {CLICKHOUSE_TABLE}"
    exists = client.execute(check_query)[0][0]
    
    if not exists:
        # Создаем таблицу с правильной структурой (как в существующей)
        create_table_query = f"""
        CREATE TABLE IF NOT EXISTS {CLICKHOUSE_TABLE} (
            city String,
            timestamp DateTime,
            temperature Float32,
            humidity UInt8,
            pressure UInt16,
            wind_speed Float32,
            precipitation Float32,
            weather_condition String
        ) ENGINE = MergeTree()
        ORDER BY (city, timestamp)
        """
        
        try:
            client.execute(create_table_query)
            print(f" Таблица {CLICKHOUSE_TABLE} создана")
        except Exception as e:
            print(f" Ошибка при создании таблицы: {e}")
    else:
        print(f" Таблица {CLICKHOUSE_TABLE} уже существует")
        
        # Покажем структуру таблицы для проверки
        desc_query = f"DESCRIBE TABLE {CLICKHOUSE_TABLE}"
        columns = client.execute(desc_query)
        print("\n Структура таблицы:")
        for col in columns:
            print(f"   {col[0]} ({col[1]})")

def fetch_openmeteo_daily(city, lat, lon, start_date, end_date):
    """
    Собирает ежедневные данные через Open-Meteo API
    Возвращает записи с timestamp (дата + 12:00 как среднее за день)
    """
    url = "https://archive-api.open-meteo.com/v1/archive"
    
    # Ограничиваем даты
    max_date = datetime.now() + timedelta(days=1)
    if end_date > max_date:
        end_date = max_date
    
    params = {
        'latitude': lat,
        'longitude': lon,
        'start_date': start_date.strftime('%Y-%m-%d'),
        'end_date': end_date.strftime('%Y-%m-%d'),
        'daily': ['temperature_2m_mean', 'relative_humidity_2m_mean', 
                  'precipitation_sum', 'pressure_msl_mean', 'wind_speed_10m_mean'],
        'timezone': 'Europe/Moscow'
    }
    
    try:
        response = requests.get(url, params=params, timeout=30)
        data = response.json()
        
        if 'error' in data:
            print(f"    Ошибка API для {city}: {data.get('reason', 'Unknown error')}")
            return []
        
        records = []
        dates = data['daily']['time']
        
        for i in range(len(dates)):
            temp = data['daily']['temperature_2m_mean'][i]
            if temp is None:
                continue
            
            # Создаем timestamp на полдень указанной даты
            dt = datetime.strptime(dates[i], '%Y-%m-%d')
            timestamp = datetime(dt.year, dt.month, dt.day, 12, 0, 0)  # 12:00 дня
            
            weather_record = {
                'city': city,
                'timestamp': timestamp,  # используем timestamp вместо date
                'temperature': temp,
                'humidity': data['daily']['relative_humidity_2m_mean'][i] or 0,
                'pressure': data['daily']['pressure_msl_mean'][i] or 1013,
                'wind_speed': (data['daily']['wind_speed_10m_mean'][i] or 0) / 3.6,
                'precipitation': data['daily']['precipitation_sum'][i] or 0,
                'weather_condition': 'historical'
            }
            records.append(weather_record)
        
        return records
        
    except Exception as e:
        print(f"    Ошибка для {city}: {e}")
        return []

def save_weather_batch(client, records):
    """Сохраняет пачку записей в ClickHouse"""
    if not records:
        return 0
    
    # Подготавливаем данные для вставки
    data_to_insert = []
    for r in records:
        try:
            data_to_insert.append((
                r['city'],
                r['timestamp'],  # datetime объект
                float(r['temperature']),
                int(r['humidity']),
                int(r['pressure']),
                float(r['wind_speed']),
                float(r['precipitation']),
                r['weather_condition']
            ))
        except Exception as e:
            print(f" Ошибка подготовки записи для {r.get('city')}: {e}")
            continue
    
    if not data_to_insert:
        return 0
    
    try:
        client.execute(
            f"INSERT INTO {CLICKHOUSE_TABLE} (city, timestamp, temperature, humidity, pressure, wind_speed, precipitation, weather_condition) VALUES",
            data_to_insert
        )
        return len(data_to_insert)
    except Exception as e:
        print(f" Ошибка сохранения: {e}")
        # Покажем пример данных для отладки
        if data_to_insert:
            print(f" Пример: {data_to_insert[0]}")
        return 0

def collect_weather_for_period(city_coords, start_date, end_date, batch_size=10):
    """
    Собирает погоду для всех городов за указанный период
    """
    print(f"\n{'='*60}")
    print(f" СБОР ПОГОДНЫХ ДАННЫХ")
    print(f"   Городов: {len(city_coords)}")
    print(f"   Период: {start_date.date()} - {end_date.date()}")
    print(f"   Дней: {(end_date - start_date).days + 1}")
    print('='*60)
    
    client = Client(
        host=CLICKHOUSE_HOST,
        port=CLICKHOUSE_PORT,
        user='default',
        password='',
        database=CLICKHOUSE_DB
    )
    
    # Создаем/проверяем таблицу
    create_weather_table()
    
    all_records = []
    successful = 0
    failed_cities = []
    total_saved = 0
    
    # Используем tqdm для прогресс-бара
    for city, coords in tqdm(city_coords.items(), desc="Обработка городов"):
        records = fetch_openmeteo_daily(
            city, 
            coords['lat'], 
            coords['lon'], 
            start_date, 
            end_date
        )
        
        if records:
            all_records.extend(records)
            successful += 1
            
            # Сохраняем каждые batch_size городов (примерно)
            if len(all_records) >= batch_size * 90:  # ~ batch_size городов по 90 дней
                saved = save_weather_batch(client, all_records)
                total_saved += saved
                print(f"\n   Промежуточно сохранено {saved} записей")
                all_records = []
        else:
            failed_cities.append(city)
        
        # Небольшая пауза между городами
        time.sleep(1)
    
    # Сохраняем остатки
    if all_records:
        saved = save_weather_batch(client, all_records)
        total_saved += saved
        print(f"\n   Финально сохранено {saved} записей")
    
    print(f"\n{'='*60}")
    print(f" СБОР ЗАВЕРШЕН:")
    print(f"   Успешно: {successful}/{len(city_coords)} городов")
    print(f"   Всего сохранено записей: {total_saved}")
    if failed_cities:
        print(f"   Провалились: {failed_cities[:10]}")
        if len(failed_cities) > 10:
            print(f"     и еще {len(failed_cities) - 10}")
    
    return successful, failed_cities, total_saved

def get_weather_stats():
    """
    Получает статистику по собранным данным
    """
    client = Client(
        host=CLICKHOUSE_HOST,
        port=CLICKHOUSE_PORT,
        user='default',
        password='',
        database=CLICKHOUSE_DB
    )
    
    print(f"\n{'='*60}")
    print(" СТАТИСТИКА ПО СОБРАННЫМ ДАННЫМ")
    print('='*60)
    
    # Общее количество
    total = client.execute(f"SELECT COUNT(*) FROM {CLICKHOUSE_TABLE}")[0][0]
    print(f" Всего записей: {total}")
    
    # По годам
    by_year = client.execute(f"""
        SELECT 
            toYear(timestamp) as year,
            COUNT(*) as cnt,
            COUNT(DISTINCT city) as cities
        FROM {CLICKHOUSE_TABLE}
        GROUP BY year
        ORDER BY year
    """)
    
    print("\n По годам:")
    for year, cnt, cities in by_year:
        print(f"   {year}: {cnt} записей, {cities} городов")
    
    # По городам
    by_city = client.execute(f"""
        SELECT 
            city,
            COUNT(*) as cnt,
            MIN(timestamp) as first,
            MAX(timestamp) as last,
            AVG(temperature) as avg_temp
        FROM {CLICKHOUSE_TABLE}
        GROUP BY city
        ORDER BY cnt DESC
        LIMIT 10
    """)
    
    print("\n Топ-10 городов по записям:")
    for city, cnt, first, last, avg_temp in by_city:
        if first and last:
            print(f"   {city}: {cnt} записей ({first.date()} - {last.date()}, ср.темп: {avg_temp:.1f}°C)")
        else:
            print(f"   {city}: {cnt} записей")

def check_existing_data(city_coords, start_date, end_date):
    """
    Проверяет, какие данные уже есть в таблице
    """
    client = Client(
        host=CLICKHOUSE_HOST,
        port=CLICKHOUSE_PORT,
        user='default',
        password='',
        database=CLICKHOUSE_DB
    )
    
    print(f"\n{'='*60}")
    print(" ПРОВЕРКА СУЩЕСТВУЮЩИХ ДАННЫХ")
    print('='*60)
    
    for city in list(city_coords.keys())[:5]:  # Проверяем первые 5 городов
        query = f"""
        SELECT 
            COUNT(*) as cnt,
            MIN(timestamp) as first,
            MAX(timestamp) as last
        FROM {CLICKHOUSE_TABLE}
        WHERE city = '{city}'
        """
        result = client.execute(query)
        if result and result[0][0] > 0:
            cnt, first, last = result[0]
            print(f"   {city}: {cnt} записей ({first.date()} - {last.date()})")
        else:
            print(f"   {city}: нет данных")

# --- ВЫПОЛНЕНИЕ ---
print("="*60)
print(" ЭТАП 2: СБОР ИСТОРИЧЕСКИХ ДАННЫХ О ПОГОДЕ")
print("="*60)

# 1. Загружаем координаты из файла
try:
    with open('city_coordinates.json', 'r', encoding='utf-8') as f:
        city_coords = json.load(f)
    print(f"\n Загружены координаты для {len(city_coords)} городов")
except FileNotFoundError:
    print("\n Файл city_coordinates.json не найден!")
    print(" Сначала выполните первую ячейку для получения координат.")
    city_coords = {}

if city_coords:
    # 2. Проверяем существующие данные
    check_existing_data(city_coords, datetime(2025, 1, 1), datetime.now())
    
    # 3. Определяем период сбора
    print(f"\n Выберите период сбора:")
    print(f"   1 - Весь 2025 год")
    print(f"   2 - Январь-Март 2025")
    print(f"   3 - Апрель-Июнь 2025")
    print(f"   4 - Июль-Сентябрь 2025")
    print(f"   5 - Октябрь-Декабрь 2025")
    print(f"   6 - 2026 год (до текущей даты)")
    print(f"   7 - Произвольный период")
    
    choice = input("\n Ваш выбор (1-7): ").strip()
    
    periods = {
        '1': (datetime(2025, 1, 1), datetime(2025, 12, 31)),
        '2': (datetime(2025, 1, 1), datetime(2025, 3, 31)),
        '3': (datetime(2025, 4, 1), datetime(2025, 6, 30)),
        '4': (datetime(2025, 7, 1), datetime(2025, 9, 30)),
        '5': (datetime(2025, 10, 1), datetime(2025, 12, 31)),
        '6': (datetime(2026, 1, 1), datetime.now()),
    }
    
    if choice in periods:
        start_date, end_date = periods[choice]
        successful, failed, total = collect_weather_for_period(
            city_coords, 
            start_date, 
            end_date,
            batch_size=10
        )
    elif choice == '7':
        start_str = input("Начальная дата (ГГГГ-ММ-ДД): ").strip()
        end_str = input("Конечная дата (ГГГГ-ММ-ДД): ").strip()
        start_date = datetime.strptime(start_str, '%Y-%m-%d')
        end_date = datetime.strptime(end_str, '%Y-%m-%d')
        successful, failed, total = collect_weather_for_period(
            city_coords, 
            start_date, 
            end_date,
            batch_size=10
        )
    else:
        print(" Неверный выбор")
    
    # 4. Показываем итоговую статистику
    get_weather_stats()

 ЭТАП 2: СБОР ИСТОРИЧЕСКИХ ДАННЫХ О ПОГОДЕ

 Загружены координаты для 125 городов

 ПРОВЕРКА СУЩЕСТВУЮЩИХ ДАННЫХ
   Адыге-Хабль аул.: 440 записей (2025-01-01 - 2026-03-16)
   Актау: 440 записей (2025-01-01 - 2026-03-16)
   Алматы: 440 записей (2025-01-01 - 2026-03-16)
   Армавир: 440 записей (2025-01-01 - 2026-03-16)
   Астрахань: 806 записей (2025-01-01 - 2026-03-16)

 Выберите период сбора:
   1 - Весь 2025 год
   2 - Январь-Март 2025
   3 - Апрель-Июнь 2025
   4 - Июль-Сентябрь 2025
   5 - Октябрь-Декабрь 2025
   6 - 2026 год (до текущей даты)
   7 - Произвольный период



 Ваш выбор (1-7):  7
Начальная дата (ГГГГ-ММ-ДД):  2024-01-01
Конечная дата (ГГГГ-ММ-ДД):  2024-12-31



 СБОР ПОГОДНЫХ ДАННЫХ
   Городов: 125
   Период: 2024-01-01 - 2024-12-31
   Дней: 366
 Таблица weather_hourly уже существует

 Структура таблицы:
   city (String)
   timestamp (DateTime('Europe/Moscow'))
   temperature (Float32)
   humidity (UInt8)
   pressure (UInt16)
   wind_speed (Float32)
   precipitation (Float32)
   weather_condition (String)
   _loaded_at (DateTime)


Обработка городов:   2%|▏         | 2/125 [00:03<03:52,  1.89s/it]


   Промежуточно сохранено 1098 записей


...


   Промежуточно сохранено 1098 записей


Обработка городов:  98%|█████████▊| 122/125 [03:42<00:05,  1.69s/it]


   Промежуточно сохранено 1098 записей


Обработка городов: 100%|██████████| 125/125 [03:52<00:00,  1.86s/it]



   Финально сохранено 732 записей

 СБОР ЗАВЕРШЕН:
   Успешно: 125/125 городов
   Всего сохранено записей: 45750

 СТАТИСТИКА ПО СОБРАННЫМ ДАННЫМ
 Всего записей: 109168

 По годам:
   2024: 45750 записей, 125 городов
   2025: 54020 записей, 125 городов
   2026: 9398 записей, 125 городов

 Топ-10 городов по записям:
   Владимир: 1172 записей (2024-01-01 - 2026-03-16, ср.темп: 6.0°C)
   Петрозаводск: 1172 записей (2024-01-01 - 2026-03-16, ср.темп: 4.5°C)
   Дзержинск: 1172 записей (2024-01-01 - 2026-03-16, ср.темп: 6.0°C)
   Симферополь: 1172 записей (2024-01-01 - 2026-03-16, ср.темп: 12.1°C)
   Волгоград: 1172 записей (2024-01-01 - 2026-03-16, ср.темп: 10.5°C)
   Ногинск: 1172 записей (2024-01-01 - 2026-03-16, ср.темп: 6.4°C)
   Ярославль: 1172 записей (2024-01-01 - 2026-03-16, ср.темп: 5.7°C)
   Махачкала: 1172 записей (2024-01-01 - 2026-03-16, ср.темп: 13.8°C)
   Лермонтов: 1172 записей (2024-01-01 - 2026-03-16, ср.темп: 10.1°C)
   Ашхабад: 1172 записей (2024-01-01 - 2026-03-16, ср.т

In [ ]:
#Функция для получения погоды за конкретную дату (для DAG)
from clickhouse_driver import Client
from datetime import datetime

def get_weather_for_date(target_date=None, cities=None):
    """
    Получает погоду для всех городов на указанную дату
    Используется в DAG для ежедневного обновления
    
    Args:
        target_date: дата (если None - сегодня)
        cities: список городов (если None - все из таблицы)
    
    Returns:
        DataFrame с погодой
    """
    if target_date is None:
        target_date = datetime.now().date()
    
    client = Client(
        host='my_clickhouse',
        port=9000,
        user='default',
        password='',
        database='external_data'
    )
    
    query = f"""
    SELECT 
        city,
        date,
        temperature,
        humidity,
        pressure,
        wind_speed,
        precipitation,
        weather_condition
    FROM weather_hourly
    WHERE date = '{target_date}'
    """
    
    if cities:
        cities_str = "', '".join(cities)
        query += f" AND city IN ('{cities_str}')"
    
    df = client.query_dataframe(query)
    print(f" Получена погода для {len(df)} городов на {target_date}")
    
    return df

# Пример использования
if __name__ == "__main__":
    # Получить погоду на сегодня
    today_weather = get_weather_for_date()
    print(today_weather.head())

In [ ]:
# В вашем DAG-файле
from weather_functions import get_weather_for_date

def update_weather(**context):
    execution_date = context['execution_date'].date()
    weather_df = get_weather_for_date(execution_date)
    # Далее мерджим с данными продаж

In [13]:
import pandas as pd

# Чтение Parquet файла (одна строка!)
df = pd.read_parquet('/home/jovyan/work/data/raw/численность населения 24-46.parquet')

# Дальше работать как с обычным DataFrame
print(df.head())
print(df.info())

   year     age    total     men   women
0  2024   0 лет  1240610  638344  602266
1  2024   1 год  1302922  669277  633645
2  2024  2 года  1392772  716930  675842
3  2024  3 года  1427114  734750  692364
4  2024  4 года  1481864  762502  719362
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2001 entries, 0 to 2000
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   year    2001 non-null   object
 1   age     2001 non-null   object
 2   total   2001 non-null   int64 
 3   men     2001 non-null   int64 
 4   women   2001 non-null   int64 
dtypes: int64(3), object(2)
memory usage: 78.3+ KB
None


In [14]:
import pandas as pd

# Чтение Parquet файла
df = pd.read_parquet('/home/jovyan/work/data/raw/численность населения 24-46.parquet')

# 1. Посмотреть общую информацию
print("📋 ИНФОРМАЦИЯ О ДАТАСЕТЕ:")
print("="*60)
print(df.info())
print("\n")

# 2. Посмотреть первые строки
print("👁 ПЕРВЫЕ 5 СТРОК:")
print("="*60)
print(df.head())
print("\n")

# 3. ПОСМОТРЕТЬ МАКСИМАЛЬНЫЙ ГОД (исправлено!)
print("📅 МАКСИМАЛЬНЫЙ ГОД:")
print("="*60)

# Ваш вариант был неверным - нужны скобки []
if 'year' in df.columns:
    max_year = df['year'].max()  # правильный синтаксис!
    print(f"Максимальный год: {max_year}")
else:
    print("Колонка 'year' не найдена")
    print(f"Доступные колонки: {df.columns.tolist()}")
print("\n")

# 4. Дополнительная статистика по годам
print("📊 СТАТИСТИКА ПО ГОДАМ:")
print("="*60)
if 'year' in df.columns:
    print(df['year'].describe())
else:
    # Если колонка называется иначе, покажем все числовые колонки
    numeric_cols = df.select_dtypes(include=['number']).columns
    for col in numeric_cols:
        print(f"\n{col}:")
        print(f"   min: {df[col].min()}")
        print(f"   max: {df[col].max()}")
        print(f"   уникальных: {df[col].nunique()}")

📋 ИНФОРМАЦИЯ О ДАТАСЕТЕ:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2001 entries, 0 to 2000
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   year    2001 non-null   object
 1   age     2001 non-null   object
 2   total   2001 non-null   int64 
 3   men     2001 non-null   int64 
 4   women   2001 non-null   int64 
dtypes: int64(3), object(2)
memory usage: 78.3+ KB
None


👁 ПЕРВЫЕ 5 СТРОК:
   year     age    total     men   women
0  2024   0 лет  1240610  638344  602266
1  2024   1 год  1302922  669277  633645
2  2024  2 года  1392772  716930  675842
3  2024  3 года  1427114  734750  692364
4  2024  4 года  1481864  762502  719362


📅 МАКСИМАЛЬНЫЙ ГОД:
Максимальный год: 2046


📊 СТАТИСТИКА ПО ГОДАМ:
count     2001
unique      23
top       2024
freq        87
Name: year, dtype: object


In [10]:
import pandas as pd
import glob
from pathlib import Path
from clickhouse_driver import Client
from datetime import datetime

# Путь к папке с Parquet файлами
data_path = '/home/jovyan/work/data/raw/*.parquet'

# Находим все parquet файлы
files = glob.glob(data_path)
print(f"Найдено файлов: {len(files)}")
for f in files:
    print(f"   - {Path(f).name}")

# Собираем все данные в один DataFrame
all_dfs = []
for file in files:
    print(f"\n Читаем: {Path(file).name}")
    df = pd.read_parquet(file)
    print(f"   Форма: {df.shape}")
    print(f"   Колонки: {df.columns.tolist()}")
    all_dfs.append(df)

# Объединяем
final_df = pd.concat(all_dfs, ignore_index=True)
print(f"\nИтоговый DataFrame: {final_df.shape}")

Найдено файлов: 1
   - численность населения 24-46.parquet

 Читаем: численность населения 24-46.parquet
   Форма: (2001, 5)
   Колонки: ['year', 'age', 'total', 'men', 'women']

Итоговый DataFrame: (2001, 5)


In [24]:
print("\n ИНФОРМАЦИЯ О ДАННЫХ:")
print("="*60)
print(final_df.info())

print("\n ПЕРВЫЕ 5 СТРОК:")
print("="*60)
print(final_df.head())

print("\n СТАТИСТИКА:")
print("="*60)
print(final_df.describe())

print("\n УНИКАЛЬНЫЕ ЗНАЧЕНИЯ В КОЛОНКАХ:")
for col in final_df.columns:
    if final_df[col].dtype == 'object':
        print(f"{col}: {final_df[col].nunique()} уникальных")
        if final_df[col].nunique() < 10:
            print(f"   {final_df[col].unique()}")


 ИНФОРМАЦИЯ О ДАННЫХ:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2001 entries, 0 to 2000
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   year    2001 non-null   object
 1   age     2001 non-null   object
 2   total   2001 non-null   int64 
 3   men     2001 non-null   int64 
 4   women   2001 non-null   int64 
dtypes: int64(3), object(2)
memory usage: 78.3+ KB
None

 ПЕРВЫЕ 5 СТРОК:
   year     age    total     men   women
0  2024   0 лет  1240610  638344  602266
1  2024   1 год  1302922  669277  633645
2  2024  2 года  1392772  716930  675842
3  2024  3 года  1427114  734750  692364
4  2024  4 года  1481864  762502  719362

 СТАТИСТИКА:
              total           men         women
count  2.001000e+03  2.001000e+03  2.001000e+03
mean   3.258755e+06  1.516413e+06  1.742342e+06
std    1.494714e+07  6.956009e+06  7.992789e+06
min    2.285570e+05  6.501600e+04  1.635410e+05
25%    1.369749e+06  6.335290e+05  7.118430e+

In [8]:
import socket

# Проверяем разные варианты
hosts_to_try = ['my_clickhouse', 'clickhouse', 'localhost', '127.0.0.1']

for host in hosts_to_try:
    try:
        print(f"\n🔍 Пробуем {host}:9000...")
        ip = socket.gethostbyname(host)
        print(f"   IP: {ip}")
        
        sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        sock.settimeout(2)
        result = sock.connect_ex((ip, 9000))
        sock.close()
        
        if result == 0:
            print(f"   ✅ Порт 9000 доступен на {host}")
        else:
            print(f"   ❌ Порт 9000 не отвечает (код: {result})")
            
    except Exception as e:
        print(f"   ❌ Ошибка резолвинга: {e}")


🔍 Пробуем my_clickhouse:9000...
   IP: 172.18.0.2
   ✅ Порт 9000 доступен на my_clickhouse

🔍 Пробуем clickhouse:9000...
   IP: 172.18.0.2
   ✅ Порт 9000 доступен на clickhouse

🔍 Пробуем localhost:9000...
   IP: 127.0.0.1
   ❌ Порт 9000 не отвечает (код: 111)

🔍 Пробуем 127.0.0.1:9000...
   IP: 127.0.0.1
   ❌ Порт 9000 не отвечает (код: 111)


In [11]:
from clickhouse_driver import Client

# Подключаемся к ClickHouse (теперь my_clickhouse работает!)
client = Client(
    host='my_clickhouse',
    port=9000,
    user='default',
    password='',
    database='external_data'
)

# Проверяем подключение
result = client.execute("SELECT 1")
print(f"✅ Подключение работает: {result}")

# Создаем таблицу для демографических данных (если ещё нет)
client.execute("""
CREATE TABLE IF NOT EXISTS demographic_data
(
    year UInt16,
    age_group String,
    total_population UInt32,
    male_population UInt32,
    female_population UInt32,
    loaded_at DateTime DEFAULT now()
) ENGINE = MergeTree()
ORDER BY (year, age_group)
""")
print("✅ Таблица demographic_data создана или уже существует")

# Предполагаем, что у вас есть final_df с данными
if 'final_df' in locals() and not final_df.empty:
    print(f"\n📊 Загружаем {len(final_df)} записей...")
    
    # Преобразуем DataFrame в список кортежей
    data_to_insert = []
    for _, row in final_df.iterrows():
        try:
            data_to_insert.append((
                int(row['year']),
                str(row['age']),
                int(row['total']),
                int(row['men']),
                int(row['women'])
            ))
        except Exception as e:
            print(f"⚠️ Ошибка в строке: {row.to_dict()} - {e}")
    
    # Загружаем пакетами по 1000 записей
    batch_size = 1000
    for i in range(0, len(data_to_insert), batch_size):
        batch = data_to_insert[i:i+batch_size]
        client.execute(
            "INSERT INTO demographic_data (year, age_group, total_population, male_population, female_population) VALUES",
            batch
        )
        print(f"✅ Загружено {min(i+batch_size, len(data_to_insert))}/{len(data_to_insert)}")
    
    # Проверяем результат
    count = client.execute("SELECT COUNT(*) FROM demographic_data")[0][0]
    print(f"\n🎉 Всего записей в таблице: {count}")
    
    # Покажем пример данных
    sample = client.execute("SELECT * FROM demographic_data LIMIT 3")
    print("\n📋 Пример загруженных данных:")
    for row in sample:
        print(f"   {row}")
        
else:
    print("❌ Нет данных в final_df")

✅ Подключение работает: [(1,)]
✅ Таблица demographic_data создана или уже существует

📊 Загружаем 2001 записей...
✅ Загружено 1000/2001
✅ Загружено 2000/2001
✅ Загружено 2001/2001

🎉 Всего записей в таблице: 2001

📋 Пример загруженных данных:
   (2024, '0 лет', 1240610, 638344, 602266, datetime.datetime(2026, 3, 13, 6, 24, 47))
   (2024, '1 год', 1302922, 669277, 633645, datetime.datetime(2026, 3, 13, 6, 24, 47))
   (2024, '10 лет', 1920588, 987037, 933551, datetime.datetime(2026, 3, 13, 6, 24, 47))


In [17]:
import pandas as pd

# Путь к файлу
file_path = '/home/jovyan/work/data/raw/sr-zpl_2025.xls'

# Читаем все листы
excel_file = pd.ExcelFile(file_path)
sheet_names = excel_file.sheet_names

print(f" Найдено листов: {len(sheet_names)}")
for i, sheet in enumerate(sheet_names, 1):
    print(f"   {i}. {sheet}")

 Найдено листов: 4
   1. Лист1
   2. Лист2
   3. Лист3
   4. Лист4


In [18]:
# Функция для просмотра листа
def preview_sheet(sheet_name, n_rows=5):
    print(f"\n{'='*60}")
    print(f"📊 Лист: {sheet_name}")
    print('='*60)
    
    df = pd.read_excel(file_path, sheet_name=sheet_name)
    print(f"Размер: {df.shape}")
    print(f"\nПервые {n_rows} строк:")
    print(df.head(n_rows))
    print(f"\nКолонки:")
    print(df.columns.tolist())
    return df

# Просматриваем каждый лист
dataframes = {}
for sheet in sheet_names:
    dataframes[sheet] = preview_sheet(sheet)


📊 Лист: Лист1
Размер: (22, 13)

Первые 5 строк:
  Средняя заработная плата по 10-процентным группам работников организаций (без субъектов малого предпринимательства)  \
0  (по данным выборочных обследований организаций...                                                                    
1                                                Год                                                                    
2                                                NaN                                                                    
3                                               2000                                                                    
4                                               2001                                                                    

  Unnamed: 1                                         Unnamed: 2 Unnamed: 3  \
0        NaN                                                NaN        NaN   
1      Всего  в том числе по 10-процентным группам работнико...     

In [19]:
from clickhouse_driver import Client

client = Client(
    host='my_clickhouse',
    port=9000,
    user='default',
    password='',
    database='external_data'
)

# 1. Таблица для зарплат по группам
client.execute("""
CREATE TABLE IF NOT EXISTS salary_groups
(
    year UInt16,
    group_name String,
    salary_value Float64,
    loaded_at DateTime DEFAULT now()
) ENGINE = MergeTree()
ORDER BY (year, group_name)
""")

# 2. Таблица для инфляции (годовое исчисление)
client.execute("""
CREATE TABLE IF NOT EXISTS inflation_yearly
(
    year UInt16,
    month UInt8,
    inflation_rate Float64,
    loaded_at DateTime DEFAULT now()
) ENGINE = MergeTree()
ORDER BY (year, month)
""")

# 3. Таблица для роста зарплат с учетом инфляции
client.execute("""
CREATE TABLE IF NOT EXISTS salary_growth
(
    year UInt16,
    growth_rate Float64,
    loaded_at DateTime DEFAULT now()
) ENGINE = MergeTree()
ORDER BY (year)
""")

# 4. Таблица для помесячной инфляции
client.execute("""
CREATE TABLE IF NOT EXISTS inflation_monthly
(
    year UInt16,
    month UInt8,
    inflation_rate Float64,
    loaded_at DateTime DEFAULT now()
) ENGINE = MergeTree()
ORDER BY (year, month)
""")

print(" Все таблицы созданы")

 Все таблицы созданы


In [20]:
for sheet in sheet_names:
    df = dataframes[sheet]
    print(f"\n{'='*60}")
    print(f" АНАЛИЗ ЛИСТА: {sheet}")
    print('='*60)
    print(f"Колонки: {df.columns.tolist()}")
    print(f"Типы данных:\n{df.dtypes}")
    print(f"\nПервые 3 строки:")
    print(df.head(3))


 АНАЛИЗ ЛИСТА: Лист1
Колонки: ['Средняя заработная плата по 10-процентным группам работников организаций (без субъектов малого предпринимательства)', 'Unnamed: 1', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4', 'Unnamed: 5', 'Unnamed: 6', 'Unnamed: 7', 'Unnamed: 8', 'Unnamed: 9', 'Unnamed: 10', 'Unnamed: 11', 'Unnamed: 12']
Типы данных:
Средняя заработная плата по 10-процентным группам работников организаций (без субъектов малого предпринимательства)    object
Unnamed: 1                                                                                                             object
Unnamed: 2                                                                                                             object
Unnamed: 3                                                                                                             object
Unnamed: 4                                                                                                             object
Unnamed: 5                         

In [21]:
import pandas as pd
from clickhouse_driver import Client
from datetime import datetime

client = Client(
    host='my_clickhouse',
    port=9000,
    user='default',
    password='',
    database='external_data'
)

# Читаем Лист2 (инфляция)
df_inflation = pd.read_excel(file_path, sheet_name='Лист2')
print("📊 Лист2 (Инфляция помесячная):")
print(df_inflation.head())

# Преобразуем в long format для ClickHouse
inflation_records = []
for _, row in df_inflation.iterrows():
    year = int(row['Год'])
    for month in range(1, 13):
        month_name = ['Янв', 'Фев', 'Мар', 'Апр', 'Май', 'Июн', 
                     'Июл', 'Авг', 'Сен', 'Окт', 'Ноя', 'Дек'][month-1]
        value = row[month_name]
        if pd.notna(value):  # пропускаем NaN
            inflation_records.append({
                'year': year,
                'month': month,
                'inflation_rate': float(value),
                'source': 'monthly'
            })

print(f"📦 Подготовлено {len(inflation_records)} записей для Лист2")

# Загружаем
if inflation_records:
    data_to_insert = [(r['year'], r['month'], r['inflation_rate']) 
                      for r in inflation_records]
    client.execute(
        "INSERT INTO inflation_monthly (year, month, inflation_rate) VALUES",
        data_to_insert
    )
    print(f"✅ Загружено {len(data_to_insert)} записей в inflation_monthly")

# Аналогично для Лист4
df_inflation_yearly = pd.read_excel(file_path, sheet_name='Лист4')
print("\n📊 Лист4 (Инфляция годовая):")
print(df_inflation_yearly.head())

inflation_yearly_records = []
for _, row in df_inflation_yearly.iterrows():
    year = int(row['Год'])
    for month in range(1, 13):
        month_name = ['Янв', 'Фев', 'Мар', 'Апр', 'Май', 'Июн', 
                     'Июл', 'Авг', 'Сен', 'Окт', 'Ноя', 'Дек'][month-1]
        value = row[month_name]
        if pd.notna(value):
            inflation_yearly_records.append({
                'year': year,
                'month': month,
                'inflation_rate': float(value),
                'source': 'yearly'
            })

if inflation_yearly_records:
    data_to_insert = [(r['year'], r['month'], r['inflation_rate']) 
                      for r in inflation_yearly_records]
    client.execute(
        "INSERT INTO inflation_yearly (year, month, inflation_rate) VALUES",
        data_to_insert
    )
    print(f"✅ Загружено {len(data_to_insert)} записей в inflation_yearly")

📊 Лист2 (Инфляция помесячная):
    Год    Янв    Фев    Мар    Апр    Май    Июн    Июл    Авг    Сен    Окт  \
0  2026   6.01    NaN    NaN    NaN    NaN    NaN    NaN    NaN    NaN    NaN   
1  2025   9.92  10.06  10.34  10.23   9.89   9.41   8.80   8.14   7.99   7.73   
2  2024   7.44   7.67   7.69   7.82   8.29   8.58   9.13   9.04   8.62   8.53   
3  2023  11.76  10.97   3.51   2.30   2.50   3.24   4.30   5.13   6.00   6.68   
4  2022   8.74   9.16  16.70  17.83  17.11  15.90  15.09  14.30  13.67  12.63   

     Ноя    Дек  Всего  
0    NaN    NaN   1.62  
1   6.65   5.60   5.60  
2   8.88   9.51   9.51  
3   7.47   7.42   7.42  
4  11.97  11.92  11.92  
📦 Подготовлено 410 записей для Лист2
✅ Загружено 410 записей в inflation_monthly

📊 Лист4 (Инфляция годовая):
    Год   Янв   Фев   Мар   Апр   Май   Июн   Июл   Авг   Сен   Окт   Ноя  \
0  2026  1.62   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   
1  2025  1.23  0.81  0.65  0.40  0.43  0.20  0.57 -0.40  0.34  0.50 

In [26]:
client.execute("DROP TABLE IF EXISTS salary_growth")

# Создаем новую таблицу с правильной структурой
client.execute("""
CREATE TABLE salary_growth
(
    year UInt16,
    indicator String,
    value Float64,
    loaded_at DateTime DEFAULT now()
) ENGINE = MergeTree()
ORDER BY (year, indicator)
""")

print("✅ Таблица salary_growth создана заново с правильной структурой")

✅ Таблица salary_growth создана заново с правильной структурой


In [27]:
def clean_salary(value):
    if pd.isna(value):
        return None
    if isinstance(value, str):
        # Заменяем все виды пробелов (обычные и неразрывные)
        cleaned = value.replace(' ', '').replace('\xa0', '').replace('\t', '')
        try:
            return float(cleaned)
        except:
            print(f"⚠️ Не удалось преобразовать: '{value}' -> '{cleaned}'")
            return None
    try:
        return float(value)
    except:
        print(f"⚠️ Не удалось преобразовать: {value}")
        return None

# Теперь загружаем Лист3 заново
df_salary_growth = pd.read_excel(file_path, sheet_name='Лист3')
print("\n📊 Лист3 (Зарплаты и рост):")
print(df_salary_growth.head())

salary_growth_records = []
for _, row in df_salary_growth.iterrows():
    year = int(row['Год'])
    actual_salary = clean_salary(row['Фактическая зарплата'])
    salary_2018 = clean_salary(row['Зарплата в ценах 2018 года'])
    
    print(f"Год {year}: факт={actual_salary}, в ценах 2018={salary_2018}")
    
    if actual_salary is not None:
        salary_growth_records.append({
            'year': year,
            'indicator': 'actual_salary',
            'value': actual_salary
        })
    if salary_2018 is not None:
        salary_growth_records.append({
            'year': year,
            'indicator': 'salary_2018_prices',
            'value': salary_2018
        })
    if pd.notna(row['Инфляция']):
        salary_growth_records.append({
            'year': year,
            'indicator': 'inflation',
            'value': float(row['Инфляция'])
        })
    if pd.notna(row['Рост зп']):
        salary_growth_records.append({
            'year': year,
            'indicator': 'salary_growth',
            'value': float(row['Рост зп'])
        })

print(f"\n📦 Подготовлено {len(salary_growth_records)} записей")

if salary_growth_records:
    # Проверим первые несколько
    print("\n👁 Пример записей:")
    for i, rec in enumerate(salary_growth_records[:5]):
        print(f"   {rec}")
    
    # Загружаем
    data_to_insert = [(r['year'], r['indicator'], r['value']) 
                      for r in salary_growth_records]
    client.execute(
        "INSERT INTO salary_growth (year, indicator, value) VALUES",
        data_to_insert
    )
    print(f"✅ Загружено {len(data_to_insert)} записей в salary_growth")


📊 Лист3 (Зарплаты и рост):
    Год Фактическая зарплата Зарплата в ценах 2018 года  Инфляция  Рост зп
0  2018               50 000                     50 000      0.06     0.08
1  2019               54 000                     50 943      0.06     0.08
2  2020               58 320                     51 905      0.06     0.08
3  2021               62 986                     52 884      0.06     0.08
4  2022               68 024                     53 882      0.06     0.08
Год 2018: факт=50000.0, в ценах 2018=50000.0
Год 2019: факт=54000.0, в ценах 2018=50943.0
Год 2020: факт=58320.0, в ценах 2018=51905.0
Год 2021: факт=62986.0, в ценах 2018=52884.0
Год 2022: факт=68024.0, в ценах 2018=53882.0
Год 2023: факт=73466.0, в ценах 2018=54898.0
Год 2024: факт=79344.0, в ценах 2018=55934.0
Год 2025: факт=85691.0, в ценах 2018=56990.0
Год 2026: факт=92547.0, в ценах 2018=58065.0
Год 2027: факт=99950.0, в ценах 2018=59160.0
Год 2028: факт=107946.0, в ценах 2018=60277.0
Год 2029: факт=116582.0, в

In [23]:
df_salary_groups = pd.read_excel(file_path, sheet_name='Лист1', header=None)
print("\n📊 Лист1 (Зарплаты по группам):")
print(df_salary_groups.head(10))

# Здесь нужно понять структуру и преобразовать
# Пропускаем первые строки с заголовками
salary_groups_records = []
for idx in range(3, len(df_salary_groups)):  # начинаем с 4-й строки
    row = df_salary_groups.iloc[idx]
    year = row[0]
    if pd.isna(year) or year in ['Год', 'NaN', None]:
        continue
    
    try:
        year_val = int(float(year))
        # Берем значения для 10 групп (колонки 2-11)
        for group_num in range(10):
            value = row[group_num + 2]  # колонки 2-11
            if pd.notna(value):
                salary_groups_records.append({
                    'year': year_val,
                    'group_name': f'group_{group_num+1}',
                    'salary_value': float(value)
                })
    except:
        continue

if salary_groups_records:
    data_to_insert = [(r['year'], r['group_name'], r['salary_value']) 
                      for r in salary_groups_records]
    client.execute(
        "INSERT INTO salary_groups (year, group_name, salary_value) VALUES",
        data_to_insert
    )
    print(f"✅ Загружено {len(data_to_insert)} записей в salary_groups")


📊 Лист1 (Зарплаты по группам):
                                                  0      1   \
0  Средняя заработная плата по 10-процентным груп...    NaN   
1  (по данным выборочных обследований организаций...    NaN   
2                                                Год  Всего   
3                                                NaN    NaN   
4                                               2000   2266   
5                                               2001   2874   
6                                               2002   4108   
7                                               2003   5017   
8                                               2004   6351   
9                                               2005   7816   

                                                  2       3       4   \
0                                                NaN     NaN     NaN   
1                                                NaN     NaN     NaN   
2  в том числе по 10-процентным группам работнико...     N

In [28]:
import pandas as pd
from clickhouse_driver import Client

# Путь к файлу
file_path = '/home/jovyan/work/data/raw/данные по муниц работникам.xls'

# Читаем и смотрим структуру
df_municipal = pd.read_excel(file_path)
print("📊 Данные по муниципальным работникам:")
print("="*60)
print(f"Размер: {df_municipal.shape}")
print(f"\nКолонки:")
print(df_municipal.columns.tolist())
print(f"\nПервые 5 строк:")
print(df_municipal.head())
print(f"\nТипы данных:")
print(df_municipal.dtypes)

📊 Данные по муниципальным работникам:
Размер: (50, 23)

Колонки:
['Средняя заработная плата работников списочного состава (рубль)', 'Unnamed: 1', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4', 'Unnamed: 5', 'Unnamed: 6', 'Unnamed: 7', 'Unnamed: 8', 'Unnamed: 9', 'Unnamed: 10', 'Unnamed: 11', 'Unnamed: 12', 'Unnamed: 13', 'Unnamed: 14', 'Unnamed: 15', 'Unnamed: 16', 'Unnamed: 17', 'Unnamed: 18', 'Unnamed: 19', 'Unnamed: 20', 'Unnamed: 21', 'Unnamed: 22']

Первые 5 строк:
  Средняя заработная плата работников списочного состава (рубль) Unnamed: 1  \
0                                                NaN                    NaN   
1                                                NaN                    NaN   
2                                                NaN                    NaN   
3                               Российская Федерация                  ВСЕГО   
4                               Российская Федерация                  ВСЕГО   

                                          Unnamed: 2   

In [29]:
import pandas as pd
import numpy as np

# Читаем файл, пропуская служебные строки
df_raw = pd.read_excel(
    '/home/jovyan/work/data/raw/данные по муниц работникам.xls',
    header=None,  # не используем первую строку как заголовок
    skiprows=3    # пропускаем первые 3 строки с пояснениями
)

print("📊 Первые 5 строк после пропуска заголовков:")
print(df_raw.head())
print("\n📋 Размер данных:", df_raw.shape)

📊 Первые 5 строк после пропуска заголовков:
                     0      1   \
0                   NaN    NaN   
1  Российская Федерация  ВСЕГО   
2  Российская Федерация  ВСЕГО   
3  Российская Федерация  ВСЕГО   
4  Российская Федерация  ВСЕГО   

                                                  2               3   \
0                                                NaN  январь-декабрь   
1  Врачи и работники медицинских организаций, име...         56444.7   
2  Младший медицинский персонал (персонал, обеспе...         21412.7   
3                                 Научные сотрудники         63430.4   
4  Педагогические работники дошкольных образовате...         29027.3   

            4            5                6               7            8   \
0  январь-март  январь-июнь  январь-сентябрь  январь-декабрь  январь-март   
1        73290        73952          73608.5         75006.8      76988.8   
2      33693.1      33634.7          33569.1         34254.3      34999.5   
3      955

In [31]:
import pandas as pd
from clickhouse_driver import Client
from datetime import datetime

# Читаем данные (уже сделано)
df_raw = pd.read_excel(
    '/home/jovyan/work/data/raw/данные по муниц работникам.xls',
    header=None,
    skiprows=3
)

# Создаем подключение к ClickHouse
client = Client(
    host='my_clickhouse',
    port=9000,
    user='default',
    password='',
    database='external_data'
)

# Создаем таблицу для муниципальных зарплат
client.execute("""
CREATE TABLE IF NOT EXISTS municipal_salaries_detailed
(
    region String,
    category String,
    year UInt16,
    period String,
    salary Float64,
    loaded_at DateTime DEFAULT now()
) ENGINE = MergeTree()
ORDER BY (region, category, year, period)
""")

print("✅ Таблица municipal_salaries_detailed создана")

# Функция для очистки чисел
def clean_number(value):
    if pd.isna(value):
        return None
    if isinstance(value, str):
        # Убираем все виды пробелов и запятые
        cleaned = value.replace(' ', '').replace('\xa0', '').replace(',', '.').replace('\t', '')
        try:
            return float(cleaned)
        except:
            return None
    return float(value)

# Определяем соответствие колонок годам и периодам
# Строка 0 содержит названия периодов для каждого года
periods_row = df_raw.iloc[0].tolist()
print("\n📅 Периоды в данных:")
for i, p in enumerate(periods_row):
    if pd.notna(p) and i >= 3:  # начиная с колонки 3
        print(f"   Колонка {i}: {p}")

# Собираем данные
records = []
# Начинаем со строки 1 (потому что строка 0 - заголовки периодов)
for idx in range(1, len(df_raw)):
    row = df_raw.iloc[idx]
    
    # Пропускаем пустые строки
    if pd.isna(row[0]) or row[0] in ['NaN', 'None', None]:
        continue
    
    region = str(row[0]) if pd.notna(row[0]) else 'Российская Федерация'
    category = str(row[2]) if pd.notna(row[2]) else 'Не указано'
    
    # Проходим по колонкам с данными (начиная с 3)
    for col in range(3, len(row)):
        value = row[col]
        if pd.notna(value) and value not in ['NaN', 'None']:
            salary = clean_number(value)
            if salary is not None:
                # Получаем период из строки 0
                period = periods_row[col] if col < len(periods_row) and pd.notna(periods_row[col]) else f'col_{col}'
                
                records.append({
                    'region': region,
                    'category': category,
                    'period': str(period),
                    'salary': salary
                })

print(f"\n📦 Подготовлено {len(records)} записей")

# Покажем пример
print("\n👁 Пример первых 10 записей:")
for i, rec in enumerate(records[:10]):
    print(f"   {i+1}. {rec}")

# Загружаем в ClickHouse
if records:
    data_to_insert = []
    for r in records:
        # Пытаемся извлечь год из периода
        year = 2020  # значение по умолчанию
        if '2017' in r['period']: year = 2017
        elif '2018' in r['period']: year = 2018
        elif '2019' in r['period']: year = 2019
        elif '2020' in r['period']: year = 2020
        elif '2021' in r['period']: year = 2021
        elif '2022' in r['period']: year = 2022
        elif '2023' in r['period']: year = 2023
        elif '2024' in r['period']: year = 2024
        elif '2025' in r['period']: year = 2025
        
        data_to_insert.append((
            r['region'],
            r['category'],
            year,
            r['period'],
            r['salary']
        ))
    
    # Загружаем пакетами по 1000
    batch_size = 1000
    for i in range(0, len(data_to_insert), batch_size):
        batch = data_to_insert[i:i+batch_size]
        client.execute(
            "INSERT INTO municipal_salaries_detailed (region, category, year, period, salary) VALUES",
            batch
        )
        print(f"✅ Загружено {min(i+batch_size, len(data_to_insert))}/{len(data_to_insert)}")
    
    # Проверяем
    count = client.execute("SELECT COUNT(*) FROM municipal_salaries_detailed")[0][0]
    print(f"\n📊 Всего записей в таблице: {count}")
    
    # Покажем пример
    sample = client.execute("SELECT * FROM municipal_salaries_detailed LIMIT 5")
    print("\n📋 Пример загруженных данных:")
    for row in sample:
        print(f"   {row}")

✅ Таблица municipal_salaries_detailed создана

📅 Периоды в данных:
   Колонка 3: январь-декабрь
   Колонка 4: январь-март
   Колонка 5: январь-июнь
   Колонка 6: январь-сентябрь
   Колонка 7: январь-декабрь
   Колонка 8: январь-март
   Колонка 9: январь-июнь
   Колонка 10: январь-сентябрь
   Колонка 11: январь-декабрь
   Колонка 12: январь-март
   Колонка 13: январь-июнь
   Колонка 14: январь-сентябрь
   Колонка 15: январь-декабрь
   Колонка 16: январь-март
   Колонка 17: январь-июнь
   Колонка 18: январь-сентябрь
   Колонка 19: январь-декабрь
   Колонка 20: январь-март
   Колонка 21: январь-июнь
   Колонка 22: январь-сентябрь

📦 Подготовлено 910 записей

👁 Пример первых 10 записей:
   1. {'region': 'Российская Федерация', 'category': 'Врачи и работники медицинских организаций, имеющие высшее медицинское (фармацевтическое) или иное высшее образование, предоставляющие медицинские услуги (обеспечивающие предоставление медицинских услуг)', 'period': 'январь-декабрь', 'salary': 56444.7}
  

In [32]:
import pandas as pd

# Путь к файлу
file_path = '/home/jovyan/work/data/raw/отношение зарплаты до 21.xls'

# Читаем и смотрим
df = pd.read_excel(file_path)
print("📊 Данные: отношение зарплаты до 21")
print("="*60)
print(f"Размер: {df.shape}")
print(f"\nКолонки:")
print(df.columns.tolist())
print(f"\nПервые 5 строк:")
print(df.head())
print(f"\nТипы данных:")
print(df.dtypes)

📊 Данные: отношение зарплаты до 21
Размер: (104, 12)

Колонки:
['Отношение среднемесячной номинальной начисленной заработной платы работников в субъекте Российской Федерации к среднероссийскому уровню', 'Unnamed: 1', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4', 'Unnamed: 5', 'Unnamed: 6', 'Unnamed: 7', 'Unnamed: 8', 'Unnamed: 9', 'Unnamed: 10', 'Unnamed: 11']

Первые 5 строк:
  Отношение среднемесячной номинальной начисленной заработной платы работников в субъекте Российской Федерации к среднероссийскому уровню  \
0                                                NaN                                                                                        
1  Классификатор объектов административно-террито...                                                                                        
2                           643 Российская Федерация                                                                                        
3                  030 Центральный федеральный округ        

In [35]:
import pandas as pd
from clickhouse_driver import Client
import re

# Читаем файл с явным указанием, что нет заголовков
df = pd.read_excel(
    '/home/jovyan/work/data/raw/отношение зарплаты до 21.xls', 
    header=None,
    dtype=str  # читаем всё как строки
)

print("📊 Первые 5 строк для понимания:")
print(df.head(10))

# Ищем строку с годами (обычно строка 1 или 2)
years_row = None
for i in range(5):  # проверим первые 5 строк
    row = df.iloc[i]
    # Ищем ячейки, которые выглядят как годы (4 цифры)
    years_found = []
    for cell in row[2:]:  # начиная с колонки 2
        cell_str = str(cell).strip()
        if cell_str.isdigit() and len(cell_str) == 4 and 2000 < int(cell_str) < 2030:
            years_found.append(int(cell_str))
    if len(years_found) > 3:  # нашли несколько годов
        years_row = i
        years = years_found
        print(f"\n✅ Годы найдены в строке {i}: {years}")
        break

if not years_row:
    print("❌ Не удалось найти годы в файле")
    # Выведем первые 5 строк для ручного анализа
    for i in range(5):
        print(f"\nСтрока {i}:")
        for j in range(min(10, df.shape[1])):
            print(f"  Колонка {j}: {df.iloc[i, j]}")
    exit()

# Подключаемся к ClickHouse
client = Client(
    host='my_clickhouse',
    port=9000,
    user='default',
    password='',
    database='external_data'
)

# Создаем таблицу
client.execute("""
CREATE TABLE IF NOT EXISTS regional_salary_ratio
(
    region_code String,
    region_name String,
    year UInt16,
    ratio_value Float64,
    loaded_at DateTime DEFAULT now()
) ENGINE = MergeTree()
ORDER BY (region_code, year)
""")

print("✅ Таблица regional_salary_ratio создана")

# Функция для очистки названия региона
def clean_region_name(name):
    if pd.isna(name) or name in ['NaN', 'None']:
        return None
    # Убираем цифровые коды в начале
    name_str = str(name).strip()
    cleaned = re.sub(r'^\d+\s*', '', name_str)
    return cleaned.strip()

# Собираем данные (начиная со строки после заголовков)
records = []
start_row = years_row + 1  # данные начинаются после строки с годами

for idx in range(start_row, len(df)):
    row = df.iloc[idx]
    
    # Проверяем, что это строка с регионом
    region_cell = str(row[0]) if pd.notna(row[0]) else ''
    
    # Пропускаем пустые строки
    if not region_cell or region_cell in ['nan', 'None', 'NaN']:
        continue
    
    # Пропускаем служебные строки
    if any(word in region_cell.lower() for word in ['классификатор', 'единица', 'период', 'процент']):
        continue
    
    # Извлекаем название региона
    # Убираем код в начале, если есть
    region_name = clean_region_name(region_cell)
    if not region_name:
        continue
    
    # Код региона (если есть)
    region_code = ''
    code_match = re.match(r'^(\d+)', str(region_cell).strip())
    if code_match:
        region_code = code_match.group(1)
    
    # Проходим по годам
    for j, year in enumerate(years):
        # Значения начинаются с колонки 2
        value_col = j + 2
        if value_col < len(row):
            value = row[value_col]
            if pd.notna(value) and value not in ['NaN', 'None']:
                try:
                    # Очищаем значение от лишних символов
                    val_str = str(value).strip().replace(',', '.')
                    ratio = float(val_str)
                    if ratio > 0:  # только положительные значения
                        records.append({
                            'region_code': region_code,
                            'region_name': region_name,
                            'year': year,
                            'ratio': ratio
                        })
                except:
                    pass

print(f"\n📦 Подготовлено {len(records)} записей")

# Покажем пример
if records:
    print("\n👁 Пример записей:")
    for i, rec in enumerate(records[:10]):
        print(f"   {i+1}. {rec}")
    
    # Загружаем в ClickHouse
    data_to_insert = [(r['region_code'], r['region_name'], r['year'], r['ratio']) 
                      for r in records]
    
    batch_size = 1000
    for i in range(0, len(data_to_insert), batch_size):
        batch = data_to_insert[i:i+batch_size]
        client.execute(
            "INSERT INTO regional_salary_ratio (region_code, region_name, year, ratio_value) VALUES",
            batch
        )
        print(f"✅ Загружено {min(i+batch_size, len(data_to_insert))}/{len(data_to_insert)}")
    
    # Проверяем
    count = client.execute("SELECT COUNT(*) FROM regional_salary_ratio")[0][0]
    print(f"\n📊 Всего записей в таблице: {count}")
    
    # Покажем пример
    sample = client.execute("SELECT * FROM regional_salary_ratio LIMIT 5")
    print("\n📋 Пример загруженных данных:")
    for row in sample:
        print(f"   {row}")
else:
    print("❌ Нет данных для загрузки")

📊 Первые 5 строк для понимания:
                                                  0                  1   \
0  Отношение среднемесячной номинальной начисленн...                NaN   
1                                                NaN                NaN   
2  Классификатор объектов административно-террито...  Единица измерения   
3                           643 Российская Федерация        744 процент   
4                  030 Центральный федеральный округ        744 процент   
5                   14000000000 Белгородская область        744 процент   
6                       15000000000 Брянская область        744 процент   
7                   17000000000 Владимирская область        744 процент   
8                    20000000000 Воронежская область        744 процент   
9                     24000000000 Ивановская область        744 процент   

                                   2      3      4      5      6      7   \
0                                 NaN    NaN    NaN    NaN    NaN 

In [36]:
import pandas as pd

# Путь к файлу
file_path = '/home/jovyan/work/data/raw/медианная зарплата по регионам до 23.xls'

# Читаем и смотрим
df = pd.read_excel(file_path, header=None)
print("📊 Медианная зарплата по регионам")
print("="*60)
print(f"Размер: {df.shape}")
print("\nПервые 10 строк для анализа:")
for i in range(min(10, len(df))):
    print(f"\nСтрока {i}:")
    for j in range(min(5, df.shape[1])):
        print(f"  Колонка {j}: {df.iloc[i, j]}")

📊 Медианная зарплата по регионам
Размер: (107, 25)

Первые 10 строк для анализа:

Строка 0:
  Колонка 0: Оплата труда наемных работников по регионам Российской Федерации
  Колонка 1: nan
  Колонка 2: nan
  Колонка 3: nan
  Колонка 4: nan

Строка 1:
  Колонка 0: nan
  Колонка 1: nan
  Колонка 2: nan
  Колонка 3: nan
  Колонка 4: nan

Строка 2:
  Колонка 0: Классификатор объектов административно-территориального деления (ОКАТО)
  Колонка 1: Единица измерения
  Колонка 2: Период
  Колонка 3: 2002.0
  Колонка 4: 2003.0

Строка 3:
  Колонка 0: 643 Российская Федерация
  Колонка 1: 385 миллион рублей
  Колонка 2: 1558883 значение показателя за год
  Колонка 3: 3445312.4
  Колонка 4: 4254262.4

Строка 4:
  Колонка 0:     643004.АГ Российская Федерация без учета новых субъектов (с 01.01.2023)
  Колонка 1: 385 миллион рублей
  Колонка 2: 1558883 значение показателя за год
  Колонка 3: nan
  Колонка 4: nan

Строка 5:
  Колонка 0:     030 Центральный федеральный округ
  Колонка 1: 385 миллион руб

In [37]:
import pandas as pd
from clickhouse_driver import Client
import re

# Читаем файл
df = pd.read_excel(
    '/home/jovyan/work/data/raw/медианная зарплата по регионам до 23.xls',
    header=None,
    dtype=str
)

# Находим строку с годами (строка 2)
years = []
for j in range(3, df.shape[1]):
    val = df.iloc[2, j]
    if pd.notna(val) and str(val).strip():
        try:
            year = int(float(val))
            if 2000 < year < 2030:
                years.append(year)
        except:
            pass

print(f"📅 Годы в данных: {years}")

# Подключаемся к ClickHouse
client = Client(
    host='my_clickhouse',
    port=9000,
    user='default',
    password='',
    database='external_data'
)

# Создаем таблицу для медианных зарплат
client.execute("""
CREATE TABLE IF NOT EXISTS median_salaries_regional
(
    region_code String,
    region_name String,
    year UInt16,
    salary_value Float64,
    loaded_at DateTime DEFAULT now()
) ENGINE = MergeTree()
ORDER BY (region_code, year)
""")

print("✅ Таблица median_salaries_regional создана")

# Функция для очистки названия региона
def clean_region_name(name):
    if pd.isna(name) or name in ['NaN', 'None']:
        return None
    name_str = str(name).strip()
    # Убираем цифровые коды в начале
    cleaned = re.sub(r'^\d+\s*', '', name_str)
    return cleaned.strip()

# Собираем данные (начиная со строки 3)
records = []
for idx in range(3, len(df)):
    row = df.iloc[idx]
    
    # Проверяем, что это строка с регионом
    region_cell = str(row[0]) if pd.notna(row[0]) else ''
    
    # Пропускаем пустые строки
    if not region_cell or region_cell in ['nan', 'None', 'NaN']:
        continue
    
    # Пропускаем служебные строки
    if any(word in region_cell.lower() for word in ['классификатор', 'единица', 'период', 'федеральный округ']):
        continue
    
    # Пропускаем итоговые строки
    if 'без учета' in region_cell.lower():
        continue
    
    # Извлекаем код региона (если есть)
    region_code = ''
    code_match = re.match(r'^(\d+)', region_cell.strip())
    if code_match:
        region_code = code_match.group(1)
    
    # Извлекаем название региона
    region_name = clean_region_name(region_cell)
    if not region_name:
        continue
    
    # Проходим по годам
    for j, year in enumerate(years):
        # Значения начинаются с колонки 3
        value_col = j + 3
        if value_col < len(row):
            value = row[value_col]
            if pd.notna(value) and value not in ['NaN', 'None']:
                try:
                    # Очищаем значение от лишних символов
                    val_str = str(value).strip().replace(' ', '').replace(',', '.')
                    salary = float(val_str)
                    if salary > 0:  # только положительные значения
                        records.append({
                            'region_code': region_code,
                            'region_name': region_name,
                            'year': year,
                            'salary': salary
                        })
                except Exception as e:
                    pass

print(f"\n📦 Подготовлено {len(records)} записей")

# Покажем пример
if records:
    print("\n👁 Пример записей (первые 10):")
    for i, rec in enumerate(records[:10]):
        print(f"   {i+1}. {rec}")
    
    # Загружаем в ClickHouse
    data_to_insert = [(r['region_code'], r['region_name'], r['year'], r['salary']) 
                      for r in records]
    
    batch_size = 1000
    for i in range(0, len(data_to_insert), batch_size):
        batch = data_to_insert[i:i+batch_size]
        client.execute(
            "INSERT INTO median_salaries_regional (region_code, region_name, year, salary_value) VALUES",
            batch
        )
        print(f"✅ Загружено {min(i+batch_size, len(data_to_insert))}/{len(data_to_insert)}")
    
    # Проверяем
    count = client.execute("SELECT COUNT(*) FROM median_salaries_regional")[0][0]
    print(f"\n📊 Всего записей в таблице: {count}")
    
    # Покажем пример загруженных данных
    sample = client.execute("SELECT * FROM median_salaries_regional LIMIT 5")
    print("\n📋 Пример загруженных данных:")
    for row in sample:
        print(f"   {row}")
else:
    print("❌ Нет данных для загрузки")

📅 Годы в данных: [2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023]
✅ Таблица median_salaries_regional создана

📦 Подготовлено 1865 записей

👁 Пример записей (первые 10):
   1. {'region_code': '643', 'region_name': 'Российская Федерация', 'year': 2002, 'salary': 3445312.4}
   2. {'region_code': '643', 'region_name': 'Российская Федерация', 'year': 2003, 'salary': 4254262.4}
   3. {'region_code': '643', 'region_name': 'Российская Федерация', 'year': 2004, 'salary': 5076191.6}
   4. {'region_code': '643', 'region_name': 'Российская Федерация', 'year': 2005, 'salary': 6291233.0}
   5. {'region_code': '643', 'region_name': 'Российская Федерация', 'year': 2006, 'salary': 7822245.0}
   6. {'region_code': '643', 'region_name': 'Российская Федерация', 'year': 2007, 'salary': 10143430.3}
   7. {'region_code': '643', 'region_name': 'Российская Федерация', 'year': 2008, 'salary': 13014684.4}
   8. {'region_code': '6

In [1]:
import pandas as pd
import os
from clickhouse_driver import Client
from datetime import datetime

# ============================================
# 1. ПРОВЕРЯЕМ СТРУКТУРУ ПЕРВОГО ФАЙЛА
# ============================================

# Путь к папке с данными (WSL путь из Windows)
data_path = '/home/jovyan/work/data/raw/'

# Список файлов
files = [
    'time_series_US_20210317-1517_20260317-1517 сыр.csv',
    'time_series_US_20210317-1516_20260317-1516 творог.csv',
    'time_series_US_20210317-1514_20260317-1514 спред.csv',
    'time_series_US_20210317-1514_20260317-1514 сливочное масло.csv',
    'time_series_US_20210317-1512_20260317-1512 молоко.csv',
    'time_series_US_20210317-1511_20260317-1511 кефир.csv',
    'time_series_US_20210317-1510_20260317-1510 йогурт.csv'
]

# Смотрим структуру первого файла
first_file = os.path.join(data_path, files[0])
print(f"📊 АНАЛИЗ ФАЙЛА: {files[0]}")
print("="*60)

df_sample = pd.read_csv(first_file)
print(f"Размер: {df_sample.shape}")
print(f"\nКолонки: {df_sample.columns.tolist()}")
print(f"\nТипы данных:")
print(df_sample.dtypes)
print(f"\nПервые 5 строк:")
print(df_sample.head())

# Проверяем наличие колонки с датой
date_columns = [col for col in df_sample.columns if 'date' in col.lower() or 'time' in col.lower()]
print(f"\n📅 Возможные колонки с датой: {date_columns}")

📊 АНАЛИЗ ФАЙЛА: time_series_US_20210317-1517_20260317-1517 сыр.csv
Размер: (61, 2)

Колонки: ['Time', 'Сыр']

Типы данных:
Time    object
Сыр      int64
dtype: object

Первые 5 строк:
         Time  Сыр
0  2021-03-01   46
1  2021-04-01   46
2  2021-05-01   43
3  2021-06-01   42
4  2021-07-01   46

📅 Возможные колонки с датой: ['Time']


In [3]:
import pandas as pd
import os
from clickhouse_driver import Client
from datetime import datetime
import glob

# ============================================
# 1. ПОДКЛЮЧАЕМСЯ К CLICKHOUSE
# ============================================

client = Client(
    host='my_clickhouse',
    port=9000,
    user='default',
    password='',
    database='external_data'
)

# ============================================
# 2. СОЗДАЕМ ТАБЛИЦУ ДЛЯ ТРЕНДОВ
# ============================================

create_table_query = """
CREATE TABLE IF NOT EXISTS google_trends_monthly (
    date Date,
    product_name String,
    trend_value UInt32,
    loaded_at DateTime DEFAULT now()
) ENGINE = MergeTree()
ORDER BY (product_name, date)
"""

client.execute(create_table_query)
print("✅ Таблица google_trends_monthly создана/проверена")

# ============================================
# 3. НАХОДИМ ВСЕ ФАЙЛЫ ПО МАСКЕ
# ============================================

data_path = '/home/jovyan/work/data/raw/'
file_pattern = os.path.join(data_path, 'time_series_US_20210317*.csv')
files = glob.glob(file_pattern)

print(f"\n📁 Найдено файлов: {len(files)}")
for f in files:
    print(f"   {os.path.basename(f)}")

# ============================================
# 4. ЗАГРУЖАЕМ ВСЕ ФАЙЛЫ
# ============================================

success_count = 0
error_count = 0

for file_path in files:
    filename = os.path.basename(file_path)
    print(f"\n📥 ЗАГРУЗКА: {filename}")
    print("="*60)
    
    try:
        # Читаем CSV
        df = pd.read_csv(file_path)
        print(f"   Прочитано строк: {len(df)}")
        print(f"   Колонки в файле: {df.columns.tolist()}")
        
        # Определяем название продукта из колонок (все кроме 'Time')
        value_columns = [col for col in df.columns if col != 'Time']
        
        if len(value_columns) == 0:
            print(f"   ❌ Не найдена колонка с данными")
            error_count += 1
            continue
        
        # Берем первую колонку с данными (обычно она одна)
        trend_col = value_columns[0]
        product_name = trend_col.strip()  # убираем лишние пробелы
        
        print(f"   Название продукта: '{product_name}'")
        
        # Преобразуем даты
        df['date'] = pd.to_datetime(df['Time'])
        
        # Подготавливаем данные для вставки
        data_to_insert = []
        for _, row in df.iterrows():
            # Проверяем, что значение не NaN
            if pd.notna(row[trend_col]):
                data_to_insert.append({
                    'date': row['date'].date(),
                    'product_name': product_name,
                    'trend_value': int(row[trend_col])
                })
        
        if data_to_insert:
            # Вставляем в ClickHouse
            client.execute(
                "INSERT INTO google_trends_monthly (date, product_name, trend_value) VALUES",
                data_to_insert
            )
            
            print(f"   ✅ Загружено {len(data_to_insert)} записей для '{product_name}'")
            success_count += 1
            
            # Покажем пример
            print(f"\n   Пример данных для '{product_name}':")
            print(df[['Time', trend_col]].head(3).to_string(index=False))
        else:
            print(f"   ❌ Нет данных для вставки")
            error_count += 1
        
    except Exception as e:
        print(f"   ❌ Ошибка при загрузке {filename}: {e}")
        error_count += 1

# ============================================
# 5. ПРОВЕРЯЕМ РЕЗУЛЬТАТ
# ============================================

print(f"\n{'='*60}")
print("📊 ИТОГИ ЗАГРУЗКИ")
print('='*60)
print(f"✅ Успешно загружено: {success_count} файлов")
print(f"❌ Ошибок: {error_count} файлов")

# Общее количество записей
result = client.execute("SELECT COUNT(*) FROM google_trends_monthly")
total_rows = result[0][0]
print(f"\nВсего записей в таблице: {total_rows}")

# Статистика по продуктам
result = client.execute("""
    SELECT 
        product_name,
        COUNT(*) as records,
        MIN(date) as first_date,
        MAX(date) as last_date,
        AVG(trend_value) as avg_trend
    FROM google_trends_monthly
    GROUP BY product_name
    ORDER BY product_name
""")

print("\n📈 Статистика по продуктам:")
for row in result:
    print(f"\n   {row[0]}:")
    print(f"      Записей: {row[1]}")
    print(f"      Период: {row[2]} - {row[3]}")
    print(f"      Средний тренд: {row[4]:.1f}")

# ============================================
# 6. ОЧИЩАЕМ ДУБЛИКАТЫ (ЕСЛИ БУДУТ)
# ============================================

print(f"\n🧹 Проверка на дубликаты...")
duplicates = client.execute("""
    SELECT product_name, date, COUNT(*) as cnt
    FROM google_trends_monthly
    GROUP BY product_name, date
    HAVING cnt > 1
""")

if duplicates:
    print(f"Найдено дубликатов: {len(duplicates)}")
    # Оставляем только последние записи
    client.execute("""
        INSERT INTO google_trends_monthly
        SELECT DISTINCT date, product_name, trend_value, now()
        FROM google_trends_monthly
    """)
else:
    print("Дубликатов нет")

✅ Таблица google_trends_monthly создана/проверена

📁 Найдено файлов: 7
   time_series_US_20210317-1514_20260317-1514 спред.csv
   time_series_US_20210317-1516_20260317-1516 творог.csv
   time_series_US_20210317-1517_20260317-1517 сыр.csv
   time_series_US_20210317-1511_20260317-1511 кефир.csv
   time_series_US_20210317-1512_20260317-1512 молоко.csv
   time_series_US_20210317-1510_20260317-1510 йогурт.csv
   time_series_US_20210317-1514_20260317-1514 сливочное масло.csv

📥 ЗАГРУЗКА: time_series_US_20210317-1514_20260317-1514 спред.csv
   Прочитано строк: 61
   Колонки в файле: ['Time', 'Намазка']
   Название продукта: 'Намазка'
   ✅ Загружено 61 записей для 'Намазка'

   Пример данных для 'Намазка':
      Time  Намазка
2021-03-01       46
2021-04-01       47
2021-05-01       46

📥 ЗАГРУЗКА: time_series_US_20210317-1516_20260317-1516 творог.csv
   Прочитано строк: 61
   Колонки в файле: ['Time', 'Кварк']
   Название продукта: 'Кварк'
   ✅ Загружено 61 записей для 'Кварк'

   Пример данны

In [4]:
import pandas as pd

# Путь к вашему файлу
excel_path = '/home/jovyan/work/data/raw/позиуии номенклатуры.xlsx'

# Читаем файл
df = pd.read_excel(excel_path)

print("📊 СТРУКТУРА ФАЙЛА")
print("="*60)
print(f"Количество строк: {len(df)}")
print(f"Количество колонок: {len(df.columns)}")
print(f"\nНазвания колонок:")
for i, col in enumerate(df.columns, 1):
    print(f"{i}. {col}")

print(f"\nТипы данных в колонках:")
print(df.dtypes)

print(f"\nПервые 5 строк:")
print(df.head())

print(f"\nПервые 5 строк (транспонировано для наглядности):")
print(df.head().T)

# Покажем несколько случайных строк
print(f"\nСлучайные 3 строки:")
print(df.sample(3).T)

# Если есть колонка с номенклатурой, покажем примеры
for col in df.columns:
    if 'номенклатур' in col.lower() or 'товар' in col.lower() or 'product' in col.lower():
        print(f"\nПримеры из колонки '{col}':")
        print(df[col].head(10).tolist())

📊 СТРУКТУРА ФАЙЛА
Количество строк: 663
Количество колонок: 3

Названия колонок:
1. №
2. номенклатура
3. категория

Типы данных в колонках:
№                int64
номенклатура    object
категория       object
dtype: object

Первые 5 строк:
   №                                      номенклатура категория
0  1           р ломт.сол.Ашан.40%.Дружба.80г.фол.20шт       сыр
1  2  Пр.пл.с сыром ломт.сол.КД.40%.Салат.80г.фол.20шт       сыр
2  3    Пр.скв.смет.ДМ.15%.350г.стак.12шт(Азербайджан)   сметана
3  4       Масса твор.с изюмом.7Утра.5%.180г.ф/пак.8шт    творог
4  5               Пр.из творога БЛ 24%.500г.пакет.8шт    творог

Первые 5 строк (транспонировано для наглядности):
                                                    0  \
№                                                   1   
номенклатура  р ломт.сол.Ашан.40%.Дружба.80г.фол.20шт   
категория                                         сыр   

                                                             1  \
№                       

In [5]:
import pandas as pd
from clickhouse_driver import Client
from datetime import datetime

client = Client(
    host='my_clickhouse',
    port=9000,
    user='default',
    password='',
    database='external_data'
)

# ============================================
# 1. ЧИТАЕМ EXCEL ФАЙЛ
# ============================================

print("📖 ЧТЕНИЕ EXCEL ФАЙЛА")
print("="*60)

excel_path = '/home/jovyan/work/data/raw/позиуии номенклатуры.xlsx'
df = pd.read_excel(excel_path)

print(f"   Загружено {len(df)} записей")
print(f"   Колонки: {df.columns.tolist()}")
print(f"\n   Уникальных категорий: {df['категория'].nunique()}")
print(df['категория'].value_counts().head(10))

# ============================================
# 2. СОЗДАЕМ ТАБЛИЦУ ДЛЯ МАППИНГА
# ============================================

print(f"\n📦 СОЗДАНИЕ ТАБЛИЦЫ product_category_mapping")
print("="*60)

client.execute("""
    CREATE TABLE IF NOT EXISTS product_category_mapping (
        product_name String,
        category String,
        loaded_at DateTime DEFAULT now()
    ) ENGINE = MergeTree()
    ORDER BY product_name
""")

# Очищаем старые данные
client.execute("TRUNCATE TABLE IF EXISTS product_category_mapping")

# ============================================
# 3. ЗАГРУЖАЕМ ДАННЫЕ
# ============================================

print(f"\n📥 ЗАГРУЗКА В CLICKHOUSE")
print("="*60)

data_to_insert = []
for _, row in df.iterrows():
    data_to_insert.append({
        'product_name': str(row['номенклатура']),
        'category': str(row['категория'])
    })

# Вставляем батчами по 1000 записей
batch_size = 1000
for i in range(0, len(data_to_insert), batch_size):
    batch = data_to_insert[i:i+batch_size]
    client.execute(
        "INSERT INTO product_category_mapping (product_name, category) VALUES",
        batch
    )
    print(f"   Загружено {min(i+batch_size, len(data_to_insert))}/{len(data_to_insert)}")

print(f"\n✅ Загружено {len(data_to_insert)} записей в product_category_mapping")

# ============================================
# 4. ПРОВЕРЯЕМ
# ============================================

print(f"\n🔍 ПРОВЕРКА")
print("="*60)

# Статистика по категориям
result = client.execute("""
    SELECT 
        category,
        COUNT(*) as cnt
    FROM product_category_mapping
    GROUP BY category
    ORDER BY cnt DESC
""")

print("\nРаспределение по категориям:")
for cat, cnt in result:
    print(f"   {cat}: {cnt} товаров")

# Проверяем несколько примеров
print(f"\nПримеры из первых 5 товаров в вашем Excel:")
for i, row in df.head(5).iterrows():
    print(f"   {row['номенклатура'][:60]}... -> {row['категория']}")

# ============================================
# 5. КАК ИСПОЛЬЗОВАТЬ В ЗАПРОСАХ
# ============================================

print(f"\n💡 ПРИМЕР ИСПОЛЬЗОВАНИЯ")
print("="*60)

# Пример запроса с присоединением категории
print("""
SELECT 
    s.`Номенклатура`,
    s.`Дата`,
    s.`Количество`,
    m.category
FROM sales_raw s
LEFT JOIN product_category_mapping m 
    ON s.`Номенклатура` = m.product_name
WHERE s.`Номенклатура` = 'Масса твор.с изюмом.7Утра.5%.180г.ф/пак.8шт'
""")

📖 ЧТЕНИЕ EXCEL ФАЙЛА
   Загружено 663 записей
   Колонки: ['№', 'номенклатура', 'категория']

   Уникальных категорий: 8
категория
творог     211
спред      123
йогурт      94
сыр         90
масло       80
сметана     48
другое       9
жир          8
Name: count, dtype: int64

📦 СОЗДАНИЕ ТАБЛИЦЫ product_category_mapping

📥 ЗАГРУЗКА В CLICKHOUSE
   Загружено 663/663

✅ Загружено 663 записей в product_category_mapping

🔍 ПРОВЕРКА

Распределение по категориям:
   творог: 211 товаров
   спред: 123 товаров
   йогурт: 94 товаров
   сыр: 90 товаров
   масло: 80 товаров
   сметана: 48 товаров
   другое: 9 товаров
   жир: 8 товаров

Примеры из первых 5 товаров в вашем Excel:
   р ломт.сол.Ашан.40%.Дружба.80г.фол.20шт... -> сыр
   Пр.пл.с сыром ломт.сол.КД.40%.Салат.80г.фол.20шт... -> сыр
   Пр.скв.смет.ДМ.15%.350г.стак.12шт(Азербайджан)... -> сметана
   Масса твор.с изюмом.7Утра.5%.180г.ф/пак.8шт... -> творог
   Пр.из творога БЛ 24%.500г.пакет.8шт... -> творог

💡 ПРИМЕР ИСПОЛЬЗОВАНИЯ

SELECT 
 

In [6]:
import pandas as pd
import os
from clickhouse_driver import Client
from datetime import datetime
import glob

client = Client(
    host='my_clickhouse',
    port=9000,
    user='default',
    password='',
    database='external_data'
)

# ============================================
# 1. НАХОДИМ ВСЕ ФАЙЛЫ С ТРЕНДАМИ
# ============================================

data_path = '/home/jovyan/work/data/raw/'
file_pattern = os.path.join(data_path, 'time_series_US_20210317*.csv')
files = glob.glob(file_pattern)

print(f"📁 Найдено файлов с трендами: {len(files)}")
for f in files:
    print(f"   {os.path.basename(f)}")

# ============================================
# 2. ЗАГРУЖАЕМ КАЖДЫЙ ФАЙЛ
# ============================================

total_loaded = 0

for file_path in files:
    filename = os.path.basename(file_path)
    print(f"\n📥 ЗАГРУЗКА: {filename}")
    
    try:
        # Читаем CSV
        df = pd.read_csv(file_path)
        print(f"   Прочитано строк: {len(df)}")
        print(f"   Колонки: {df.columns.tolist()}")
        
        # Определяем название продукта (колонка с данными)
        value_cols = [col for col in df.columns if col != 'Time']
        
        if not value_cols:
            print(f"   ❌ Не найдена колонка с данными")
            continue
        
        product_name = value_cols[0]
        print(f"   Продукт: '{product_name}'")
        
        # Подготавливаем данные
        data_to_insert = []
        for _, row in df.iterrows():
            if pd.notna(row[product_name]):
                data_to_insert.append({
                    'date': pd.to_datetime(row['Time']).date(),
                    'product_name': product_name,
                    'trend_value': int(row[product_name])
                })
        
        if data_to_insert:
            # Вставляем в ClickHouse
            client.execute(
                "INSERT INTO google_trends_monthly (date, product_name, trend_value) VALUES",
                data_to_insert
            )
            print(f"   ✅ Загружено {len(data_to_insert)} записей для '{product_name}'")
            total_loaded += len(data_to_insert)
        else:
            print(f"   ❌ Нет данных для вставки")
            
    except Exception as e:
        print(f"   ❌ Ошибка: {e}")

print(f"\n{'='*60}")
print(f"✅ ВСЕГО ЗАГРУЖЕНО: {total_loaded} записей")
print('='*60)

📁 Найдено файлов с трендами: 8
   time_series_US_20210317-1514_20260317-1514 спред.csv
   time_series_US_20210317-1516_20260317-1516 творог.csv
   time_series_US_20210317-1517_20260317-1517 сыр.csv
   time_series_US_20210317-1511_20260317-1511 кефир.csv
   time_series_US_20210317-1536_20260317-1536.csv
   time_series_US_20210317-1512_20260317-1512 молоко.csv
   time_series_US_20210317-1510_20260317-1510 йогурт.csv
   time_series_US_20210317-1514_20260317-1514 сливочное масло.csv

📥 ЗАГРУЗКА: time_series_US_20210317-1514_20260317-1514 спред.csv
   Прочитано строк: 61
   Колонки: ['Time', 'Намазка']
   Продукт: 'Намазка'
   ✅ Загружено 61 записей для 'Намазка'

📥 ЗАГРУЗКА: time_series_US_20210317-1516_20260317-1516 творог.csv
   Прочитано строк: 61
   Колонки: ['Time', 'Кварк']
   Продукт: 'Кварк'
   ✅ Загружено 61 записей для 'Кварк'

📥 ЗАГРУЗКА: time_series_US_20210317-1517_20260317-1517 сыр.csv
   Прочитано строк: 61
   Колонки: ['Time', 'Сыр']
   Продукт: 'Сыр'
   ✅ Загружено 61 запи

In [7]:
# Проверяем статистику по трендам
result = client.execute("""
    SELECT 
        product_name,
        COUNT(*) as records,
        MIN(date) as first_date,
        MAX(date) as last_date,
        AVG(trend_value) as avg_trend
    FROM google_trends_monthly
    GROUP BY product_name
    ORDER BY product_name
""")

print("\n📊 СТАТИСТИКА ПО ТРЕНДАМ:")
print("="*60)
for row in result:
    print(f"\n{row[0]}:")
    print(f"   Записей: {row[1]}")
    print(f"   Период: {row[2]} - {row[3]}")
    print(f"   Средний тренд: {row[4]:.1f}")


📊 СТАТИСТИКА ПО ТРЕНДАМ:

Кварк:
   Записей: 183
   Период: 2021-03-01 - 2026-03-01
   Средний тренд: 64.4

Молоко:
   Записей: 183
   Период: 2021-03-01 - 2026-03-01
   Средний тренд: 74.3

Намазка:
   Записей: 183
   Период: 2021-03-01 - 2026-03-01
   Средний тренд: 61.0

Сливочное масло:
   Записей: 183
   Период: 2021-03-01 - 2026-03-01
   Средний тренд: 59.0

Сметана:
   Записей: 61
   Период: 2021-03-01 - 2026-03-01
   Средний тренд: 58.7

Сыр:
   Записей: 183
   Период: 2021-03-01 - 2026-03-01
   Средний тренд: 66.6

йогурт:
   Записей: 244
   Период: 2021-03-01 - 2026-03-01
   Средний тренд: 26.1

кефир:
   Записей: 244
   Период: 2021-03-01 - 2026-03-01
   Средний тренд: 42.3


In [8]:
from clickhouse_driver import Client
from datetime import datetime

client = Client(
    host='my_clickhouse',
    port=9000,
    user='default',
    password='',
    database='external_data'
)

# ============================================
# 1. СМОТРИМ, ЧТО УЖЕ ЕСТЬ
# ============================================

print("📊 ТЕКУЩАЯ СИТУАЦИЯ В google_trends_monthly:")
print("="*60)

duplicates = client.execute("""
    SELECT 
        product_name,
        COUNT(*) as cnt,
        MIN(date) as first_date,
        MAX(date) as last_date
    FROM google_trends_monthly
    GROUP BY product_name
    ORDER BY product_name
""")

for row in duplicates:
    print(f"\n{row[0]}:")
    print(f"   Записей: {row[1]}")
    print(f"   Период: {row[2]} - {row[3]}")

# ============================================
# 2. ОЧИЩАЕМ ДУБЛИКАТЫ (оставляем последние загрузки)
# ============================================

print(f"\n🧹 ОЧИСТКА ДУБЛИКАТОВ...")

# Создаем временную таблицу с уникальными записями
client.execute("""
    CREATE TEMPORARY TABLE trends_unique AS
    SELECT DISTINCT date, product_name, trend_value
    FROM google_trends_monthly
""")

# Очищаем основную таблицу
client.execute("TRUNCATE TABLE google_trends_monthly")

# Вставляем уникальные записи обратно
client.execute("""
    INSERT INTO google_trends_monthly (date, product_name, trend_value)
    SELECT date, product_name, trend_value FROM trends_unique
""")

print("✅ Дубликаты удалены")

# ============================================
# 3. ПРОВЕРЯЕМ РЕЗУЛЬТАТ
# ============================================

print(f"\n📊 ПОСЛЕ ОЧИСТКИ:")

clean_stats = client.execute("""
    SELECT 
        product_name,
        COUNT(*) as cnt,
        MIN(date) as first_date,
        MAX(date) as last_date
    FROM google_trends_monthly
    GROUP BY product_name
    ORDER BY product_name
""")

total = 0
for row in clean_stats:
    print(f"\n{row[0]}: {row[1]} записей ({row[2]} - {row[3]})")
    total += row[1]

print(f"\n✅ ВСЕГО ЗАПИСЕЙ: {total} (должно быть 8 продуктов × 61 месяц = 488)")

# ============================================
# 4. СОЗДАЕМ ПРАВИЛЬНЫЙ МАППИНГ КАТЕГОРИЙ
# ============================================

print(f"\n🔗 СОЗДАНИЕ МАППИНГА КАТЕГОРИЙ ДЛЯ ТРЕНДОВ...")

# Словарь соответствия: ваша категория -> название в трендах
category_to_trend = {
    'творог': 'Кварк',      # проверьте, может быть 'творог'?
    'сыр': 'Сыр',
    'молоко': 'Молоко',
    'йогурт': 'йогурт',
    'кефир': 'кефир',
    'сметана': 'Сметана',
    'спред': 'Намазка',
    'сливочное масло': 'Сливочное масло'
}

# Добавляем колонку trend_name в product_category_mapping
client.execute("""
    ALTER TABLE product_category_mapping 
    ADD COLUMN IF NOT EXISTS trend_name String
""")

# Обновляем trend_name для каждой категории
for category, trend in category_to_trend.items():
    client.execute(
        "ALTER TABLE product_category_mapping UPDATE trend_name = %(trend)s WHERE category = %(category)s",
        {'trend': trend, 'category': category}
    )

print("✅ Маппинг обновлен")

# ============================================
# 5. ПРОВЕРЯЕМ МАППИНГ
# ============================================

mapping_check = client.execute("""
    SELECT 
        category,
        trend_name,
        COUNT(*) as cnt
    FROM product_category_mapping
    WHERE trend_name IS NOT NULL
    GROUP BY category, trend_name
    ORDER BY category
""")

print(f"\n📋 МАППИНГ КАТЕГОРИЙ:")
for cat, trend, cnt in mapping_check:
    print(f"   {cat:15} -> {trend:15} ({cnt} товаров)")

📊 ТЕКУЩАЯ СИТУАЦИЯ В google_trends_monthly:

Кварк:
   Записей: 183
   Период: 2021-03-01 - 2026-03-01

Молоко:
   Записей: 183
   Период: 2021-03-01 - 2026-03-01

Намазка:
   Записей: 183
   Период: 2021-03-01 - 2026-03-01

Сливочное масло:
   Записей: 183
   Период: 2021-03-01 - 2026-03-01

Сметана:
   Записей: 61
   Период: 2021-03-01 - 2026-03-01

Сыр:
   Записей: 183
   Период: 2021-03-01 - 2026-03-01

йогурт:
   Записей: 244
   Период: 2021-03-01 - 2026-03-01

кефир:
   Записей: 244
   Период: 2021-03-01 - 2026-03-01

🧹 ОЧИСТКА ДУБЛИКАТОВ...
✅ Дубликаты удалены

📊 ПОСЛЕ ОЧИСТКИ:

Кварк: 61 записей (2021-03-01 - 2026-03-01)

Молоко: 61 записей (2021-03-01 - 2026-03-01)

Намазка: 61 записей (2021-03-01 - 2026-03-01)

Сливочное масло: 61 записей (2021-03-01 - 2026-03-01)

Сметана: 61 записей (2021-03-01 - 2026-03-01)

Сыр: 61 записей (2021-03-01 - 2026-03-01)

йогурт: 61 записей (2021-03-01 - 2026-03-01)

кефир: 61 записей (2021-03-01 - 2026-03-01)

✅ ВСЕГО ЗАПИСЕЙ: 488 (должно быть

In [9]:
from clickhouse_driver import Client
import pandas as pd

client = Client(
    host='my_clickhouse',
    port=9000,
    user='default',
    password='',
    database='external_data'
)

# ============================================
# 1. СОЗДАЕМ ПРАВИЛЬНЫЙ МАППИНГ В PYTHON
# ============================================

print("🔗 СОЗДАНИЕ МАППИНГА КАТЕГОРИЙ...")

# Словарь соответствия: ваша категория -> название в трендах
category_to_trend = {
    'творог': 'Кварк',
    'сыр': 'Сыр',
    'молоко': 'Молоко',
    'йогурт': 'йогурт',
    'кефир': 'кефир',
    'сметана': 'Сметана',
    'спред': 'Намазка',
    'сливочное масло': 'Сливочное масло'
}

# ============================================
# 2. ЗАГРУЖАЕМ ТЕКУЩИЙ МАППИНГ
# ============================================

mapping_df = client.query_dataframe("SELECT product_name, category FROM product_category_mapping")
print(f"\n📋 ТЕКУЩИЙ МАППИНГ:")
print(mapping_df['category'].value_counts())

# ============================================
# 3. ДОБАВЛЯЕМ КОЛОНКУ trend_name (если нет)
# ============================================

try:
    client.execute("ALTER TABLE product_category_mapping ADD COLUMN IF NOT EXISTS trend_name String")
    print("✅ Колонка trend_name добавлена")
except Exception as e:
    print(f"⚠️ Колонка уже существует: {e}")

# ============================================
# 4. СОЗДАЕМ ВРЕМЕННУЮ ТАБЛИЦУ С ПРАВИЛЬНЫМИ ДАННЫМИ
# ============================================

print("\n🔄 СОЗДАНИЕ ПРАВИЛЬНОГО МАППИНГА...")

# Создаем временную таблицу
client.execute("""
    CREATE TEMPORARY TABLE temp_mapping (
        product_name String,
        category String,
        trend_name String
    )
""")

# Подготавливаем данные
data_to_insert = []
for _, row in mapping_df.iterrows():
    trend = category_to_trend.get(row['category'], '')
    data_to_insert.append({
        'product_name': row['product_name'],
        'category': row['category'],
        'trend_name': trend
    })

# Вставляем во временную таблицу
client.execute(
    "INSERT INTO temp_mapping (product_name, category, trend_name) VALUES",
    data_to_insert
)

# ============================================
# 5. ОБНОВЛЯЕМ ОСНОВНУЮ ТАБЛИЦУ
# ============================================

# Очищаем основную таблицу
client.execute("TRUNCATE TABLE product_category_mapping")

# Вставляем обновленные данные
client.execute("""
    INSERT INTO product_category_mapping (product_name, category, trend_name)
    SELECT product_name, category, trend_name FROM temp_mapping
""")

print("✅ Маппинг обновлен")

# ============================================
# 6. ПРОВЕРЯЕМ РЕЗУЛЬТАТ
# ============================================

result = client.execute("""
    SELECT 
        trend_name,
        COUNT(*) as cnt
    FROM product_category_mapping
    GROUP BY trend_name
    ORDER BY cnt DESC
""")

print("\n📊 МАППИНГ ПОСЛЕ ОБНОВЛЕНИЯ:")
for trend, cnt in result:
    print(f"   {trend if trend else 'Не указано':15}: {cnt} товаров")

# Покажем примеры для каждой категории
print("\n📋 ПРИМЕРЫ ДЛЯ КАЖДОЙ КАТЕГОРИИ:")
for category in category_to_trend.keys():
    samples = client.execute(f"""
        SELECT product_name, trend_name 
        FROM product_category_mapping 
        WHERE category = '{category}'
        LIMIT 2
    """)
    print(f"\n{category}:")
    for prod, trend in samples:
        print(f"   {prod[:50]}... -> {trend}")

🔗 СОЗДАНИЕ МАППИНГА КАТЕГОРИЙ...

📋 ТЕКУЩИЙ МАППИНГ:
category
творог     211
спред      123
йогурт      94
сыр         90
масло       80
сметана     48
другое       9
жир          8
Name: count, dtype: int64
✅ Колонка trend_name добавлена

🔄 СОЗДАНИЕ ПРАВИЛЬНОГО МАППИНГА...
✅ Маппинг обновлен

📊 МАППИНГ ПОСЛЕ ОБНОВЛЕНИЯ:
   Кварк          : 211 товаров
   Намазка        : 123 товаров
   Не указано     : 97 товаров
   йогурт         : 94 товаров
   Сыр            : 90 товаров
   Сметана        : 48 товаров

📋 ПРИМЕРЫ ДЛЯ КАЖДОЙ КАТЕГОРИИ:

творог:
   Десерт БЛ.7,2%.с абрик.130г.ст.8шт... -> Кварк
   Десерт БЛ.7,2%.с абрик.130г.ст.8шт(Азербайджан)... -> Кварк

сыр:
   Мол.пр.плав.сыр.АБИ.40%.колб.1кг.пл.8шт... -> Сыр
   Пл.пр.БЛ.45%.ШАР.Российск.400г.8шт... -> Сыр

молоко:

йогурт:
   Биойогурт.7Утра.2,5%.150г.ст.8шт... -> йогурт
   Биойогурт.7Утра.2,5%.с вишн.150г.ст.8шт... -> йогурт

кефир:

сметана:
   Мол.смет.пр.БЛ.Классическая.10%.3кг.вед.№20... -> Сметана
   Мол.смет.пр.БЛ.Классич

In [ ]:
from clickhouse_driver import Client

# Создаем подключение заново
client = Client(
    host='my_clickhouse',
    port=9000,
    user='default',
    password='',
    database='external_data'
)

# Теперь проверяем
result = client.execute("SELECT COUNT(*) FROM external_data.weather_hourly")
print(f" Всего записей в таблице: {result[0][0]}")

# Посмотрим первые несколько записей
sample = client.execute("SELECT city, timestamp, temperature FROM external_data.weather_hourly LIMIT 5")
print("\nПример данных:")
for row in sample:
    print(f"   {row}")

In [6]:
import os
import pathlib

path = r"\\vra.local\Root\RF_Downloads\user0907\Downloads\20260326\20260326\batch1\full_city_analysis_20260326_131248.txt"

# Проверить, существует ли файл
print(f"Файл существует: {os.path.exists(path)}")

# Если True, читаем
if os.path.exists(path):
    with open(path, "rb") as f:
        first_bytes = f.read(4)
        print(list(first_bytes))

Файл существует: False
